# Project Setup for Colab and Kaggle

This notebook was automatically bundled for cloud execution. Run the cell below to reconstruct the project structure and install dependencies.


In [17]:
# =========================================================
# CLOUD ENVIRONMENT SETUP (AUTO-GENERATED)
# =========================================================
import os
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ

if IN_COLAB or IN_KAGGLE:
    print("Running in Cloud Environment")
    
    
    
    # Copy dataset
    print("Copying dataset...")
    !apt -qq install rclone && rclone copy /kaggle/input/notebooks/mustafamuhaimin/3d-human-motion-generation/ /kaggle/working/ --transfers 16 --checkers 16 --progress --ignore-existing -q --exclude "dataset/**" --checksum
    
    # Install dependencies
    print("Installing dependencies (this may take a minute)...")
    %pip install -r requirements.txt
    
    print("Setup Complete!")
else:
    print("Running locally. No setup needed.")


In [18]:
!rm -rf config.py models.py utils/ && ls -lah .

In [19]:
FILES = {
    "config.py": "from pathlib import Path\nfrom dataclasses import dataclass\nfrom typing import Optional, List, Tuple\n\n@dataclass\nclass Config:\n    device: str = 'cuda'\n    seed: int = 42\n    dataset_path: Path = Path('./dataset/humanml3d-subset')\n    output_path: Path = Path('./generation')\n    checkpoint_dir: Path = Path('./checkpoints')\n    motion_dim: int = 271\n    num_joints: int = 22\n    joint_dim: int = 3\n    max_motion_length: int = 200\n    fps: int = 20\n    feature_dims: tuple[slice, ...] = (slice(0, 3), slice(3, 69), slice(69, 201), slice(201, 267), slice(267, 271))\n    dataset_name: str = 't2m'\n    unit_length: int = 5\n    text_embedding_dim: int = 512\n    per_joint_out_dim: int = 64\n    max_text_seq_len: int = 1\n    model_dim: int = 256\n    num_encoder_layers: int = 4\n    dropout: float = 0.1\n    num_flow_layers: int = 4\n    num_heads: int = 4\n    time_embed_dim: int = 64\n    batch_size: int = 200\n    learning_rate: float = 0.0001\n    num_epochs: int = 200\n    weight_decay: float = 1e-05\n    gradient_clip: float = 1.0\n    warmup_steps: int = 1000\n    lr_decay: float = 0.95\n    lr_decay_epoch: int = 10\n    flow_loss_weight: float = 1.0\n    context_loss_weight: float = 0.1\n    num_inference_steps: int = 50\n    guidance_scale: float = 1.0\n    num_workers: int = 4\n    pin_memory: bool = True\n    log_interval: int = 50\n    save_interval: int = 5\n    eval_interval: int = 1\n    num_eval_samples: int = 100\n    eval_batch_size: int = 32\n\n    def __post_init__(self):\n        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)\n        self.output_path.mkdir(parents=True, exist_ok=True)\n        self.dataset_path.mkdir(parents=True, exist_ok=True)\n\n    def to_dict(self) -> dict:\n        return {k: str(v) if isinstance(v, Path) else v for k, v in self.__dict__.items()}",
    "models.py": "import torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom pathlib import Path\nfrom typing import Optional, List, Tuple, Union, Callable, Any\nfrom config import Config\nfrom utils.motion_utils import features_to_positions, preprocess_sequence, get_dataset_config, flow_output_to_positions\n\nclass Rotary(nn.Module):\n\n    def __init__(self, dim, base=10000):\n        super().__init__()\n        inv_freq = 1.0 / base ** (torch.arange(0, dim, 2).float() / dim)\n        self.register_buffer('inv_freq', inv_freq)\n        self.seq_len_cached = None\n        self.cos_cached = None\n        self.sin_cached = None\n\n    def forward(self, x):\n        seq_len = x.shape[2]\n        if seq_len != self.seq_len_cached:\n            self.seq_len_cached = seq_len\n            t = torch.arange(seq_len, device=x.device).type_as(self.inv_freq)\n            freqs = torch.einsum('i,j->ij', t, self.inv_freq)\n            emb = torch.cat((freqs, freqs), dim=-1)\n            self.cos_cached = emb.cos()[None, None, :, :]\n            self.sin_cached = emb.sin()[None, None, :, :]\n        return (self.cos_cached, self.sin_cached)\n\ndef rotate_half(x):\n    x1, x2 = (x[..., :x.shape[-1] // 2], x[..., x.shape[-1] // 2:])\n    return torch.cat((-x2, x1), dim=-1)\n\ndef apply_rotary_pos_emb(q, k, cos, sin):\n    q = q * cos + rotate_half(q) * sin\n    k = k * cos + rotate_half(k) * sin\n    return (q, k)\n\nclass TemporalAttentionWithRoPE(nn.Module):\n\n    def __init__(self, model_dim: int, nhead: int, dropout: float=0.1):\n        super().__init__()\n        assert model_dim % nhead == 0, 'model_dim must be divisible by nhead'\n        self.model_dim = model_dim\n        self.nhead = nhead\n        self.head_dim = model_dim // nhead\n        self.qkv_proj = nn.Linear(model_dim, model_dim * 3)\n        self.out_proj = nn.Linear(model_dim, model_dim)\n        self.rope = Rotary(self.head_dim)\n        self.dropout = dropout\n\n    def forward(self, x, causal_mask=None):\n        B, T, D = x.shape\n        qkv = self.qkv_proj(x)\n        qkv = qkv.view(B, T, 3, self.nhead, self.head_dim)\n        qkv = qkv.permute(2, 0, 3, 1, 4)\n        q, k, v = qkv.unbind(0)\n        cos, sin = self.rope(q)\n        q, k = apply_rotary_pos_emb(q, k, cos, sin)\n        attn_output = F.scaled_dot_product_attention(q, k, v, dropout_p=self.dropout if self.training else 0.0, is_causal=True)\n        attn_output = attn_output.transpose(1, 2).contiguous()\n        attn_output = attn_output.view(B, T, D)\n        out = self.out_proj(attn_output)\n        return out\n\nclass TemporalTransformerBlock(nn.Module):\n\n    def __init__(self, model_dim, nhead, dropout=0.1):\n        super().__init__()\n        self.norm1 = nn.LayerNorm(model_dim)\n        self.attn = TemporalAttentionWithRoPE(model_dim, nhead, dropout)\n        self.norm2 = nn.LayerNorm(model_dim)\n        self.ffn = nn.Sequential(nn.Linear(model_dim, model_dim * 4), nn.GELU(), nn.Dropout(dropout), nn.Linear(model_dim * 4, model_dim))\n        self.dropout = nn.Dropout(dropout)\n\n    def forward(self, x, causal_mask=None):\n        x = x + self.dropout(self.attn(self.norm1(x), causal_mask))\n        x = x + self.dropout(self.ffn(self.norm2(x)))\n        return x\n\nclass SpatiotemporalBlock(nn.Module):\n\n    def __init__(self, model_dim, nhead, dropout):\n        super().__init__()\n        self.spatial_norm1 = nn.LayerNorm(model_dim)\n        self.spatial_attn = nn.MultiheadAttention(embed_dim=model_dim, num_heads=nhead, dropout=dropout, batch_first=True)\n        self.spatial_dropout = nn.Dropout(dropout)\n        self.spatial_norm2 = nn.LayerNorm(model_dim)\n        self.spatial_ffn = nn.Sequential(nn.Linear(model_dim, model_dim * 4), nn.GELU(), nn.Dropout(dropout), nn.Linear(model_dim * 4, model_dim))\n        self.temporal_block = TemporalTransformerBlock(model_dim=model_dim, nhead=nhead, dropout=dropout)\n\n    def forward(self, x):\n        B, T, S, D = x.shape\n        x_spatial = x.reshape(B * T, S, D)\n        residual = x_spatial\n        x_spatial = self.spatial_norm1(x_spatial)\n        attn_out, _ = self.spatial_attn(x_spatial, x_spatial, x_spatial, need_weights=False)\n        x_spatial = residual + self.spatial_dropout(attn_out)\n        residual = x_spatial\n        x_spatial = self.spatial_norm2(x_spatial)\n        x_spatial = residual + self.spatial_dropout(self.spatial_ffn(x_spatial))\n        x = x_spatial.reshape(B, T, S, D)\n        x = x.transpose(1, 2)\n        x_temporal = x.reshape(B * S, T, D)\n        x_temporal = self.temporal_block(x_temporal)\n        x = x_temporal.reshape(B, S, T, D)\n        x = x.transpose(1, 2)\n        return x\n\nclass MotionHistoryEncoder(nn.Module):\n\n    def __init__(self, frame_feature_dim: int, text_embedding_dim: int, per_joint_out_dim: int, joint_count: int=22, model_dim: int=256, num_layers: int=4, max_text_seq_len: int=1, dropout: float=0.1) -> None:\n        super().__init__()\n        self.frame_feature_dim = frame_feature_dim\n        self.text_embedding_dim = text_embedding_dim\n        self.per_joint_out_dim = per_joint_out_dim\n        self.model_dim = model_dim\n        self.num_layers = num_layers\n        self.joint_count = joint_count\n        self.max_text_seq_len = max_text_seq_len\n        self.nhead = model_dim // 64\n        self.head_dim = model_dim // self.nhead\n        assert self.head_dim * self.nhead == model_dim, 'model_dim must be divisible by nhead'\n        self.text_projection = nn.Linear(text_embedding_dim, model_dim)\n        self.global_feature_dim = 16\n        self.global_proj = nn.Linear(self.global_feature_dim, model_dim)\n        self.track_feature_dim = 12\n        self.track_proj = nn.Linear(self.track_feature_dim, model_dim)\n        self.kinematic_encoder = KinematicChainEncoder(model_dim)\n        self.blocks = nn.ModuleList([SpatiotemporalBlock(model_dim=model_dim, nhead=self.nhead, dropout=dropout) for _ in range(num_layers)])\n        self.final_norm = nn.LayerNorm(model_dim)\n        self.dropout = nn.Dropout(dropout)\n        self.output_proj = nn.Linear(model_dim, per_joint_out_dim)\n        self.null_history = nn.Parameter(torch.zeros(1, 1, frame_feature_dim))\n        self.null_text_embedding = nn.Parameter(torch.zeros(1, max_text_seq_len, text_embedding_dim))\n        self._init_weights()\n\n    def _init_weights(self):\n        for module in self.modules():\n            if isinstance(module, nn.Linear):\n                nn.init.xavier_uniform_(module.weight)\n                if module.bias is not None:\n                    nn.init.zeros_(module.bias)\n            elif isinstance(module, nn.Embedding):\n                nn.init.xavier_uniform_(module.weight)\n\n    def forward(self, text: Optional[torch.Tensor], input_features: Optional[torch.Tensor]=None, total_duration: Optional[torch.Tensor]=None, batch_size: Optional[int]=None, return_all_timesteps: bool=False) -> torch.Tensor:\n        if input_features is not None:\n            B = input_features.shape[0]\n        elif text is not None:\n            B = text.shape[0]\n        elif batch_size is not None:\n            B = batch_size\n        else:\n            B = 1\n        device = next(self.parameters()).device\n        if text is None:\n            text = self.null_text_embedding.to(device).expand(B, -1, -1)\n        elif text.shape[0] != B:\n            if text.shape[0] == 1:\n                text = text.expand(B, -1, -1)\n            else:\n                raise ValueError(f'Batch mismatch: text tensor({text.shape[0]}) vs batch({B})')\n        L_text = text.shape[1]\n        if input_features is None:\n            T = 1\n            input_features = self.null_history.to(device).expand(B, T, -1)\n        else:\n            T = input_features.shape[1]\n        S = self.joint_count\n        text_tokens = self.text_projection(text)\n        text_tokens = text_tokens.unsqueeze(2).expand(B, L_text, S, self.model_dim)\n        root_height_y = input_features[:, :, 0:1]\n        root_vel_x = input_features[:, :, 1:2]\n        root_vel_z = input_features[:, :, 2:3]\n        root_rot_6d = input_features[:, :, 69:75]\n        root_local_vel = input_features[:, :, 201:204]\n        foot_contacts = input_features[:, :, 267:271]\n        global_features = torch.cat([root_height_y, root_vel_x, root_vel_z, root_rot_6d, root_local_vel, foot_contacts], dim=-1)\n        global_token = self.global_proj(global_features)\n        global_token = global_token.unsqueeze(2)\n        ric = input_features[:, :, 6:69].view(B, T, 21, 3)\n        rot = input_features[:, :, 75:201].view(B, T, 21, 6)\n        vel = input_features[:, :, 204:267].view(B, T, 21, 3)\n        track_features = torch.cat([ric, rot, vel], dim=-1)\n        track_tokens = self.track_proj(track_features)\n        motion_tokens = torch.cat([global_token, track_tokens], dim=2)\n        joint_ids = torch.arange(self.joint_count, device=device)\n        kinematic_emb = self.kinematic_encoder(joint_ids)\n        motion_tokens = motion_tokens + kinematic_emb.unsqueeze(0).unsqueeze(0)\n        x = torch.cat([text_tokens, motion_tokens], dim=1)\n        x = self.dropout(x)\n        for block in self.blocks:\n            x = block(x)\n        x = self.final_norm(x)\n        x = x[:, -T:]\n        out = self.output_proj(x)\n        return out\n\n    @property\n    def output_dim(self) -> int:\n        return self.per_joint_out_dim\n\nclass KinematicChainEncoder(nn.Module):\n\n    def __init__(self, model_dim: int) -> None:\n        super().__init__()\n        joint_to_chain = [0] * 22\n        joint_to_depth = [0] * 22\n        for d, j in enumerate([0, 2, 5, 8, 11]):\n            joint_to_chain[j], joint_to_depth[j] = (0, d)\n        for d, j in enumerate([1, 4, 7, 10], 1):\n            joint_to_chain[j], joint_to_depth[j] = (1, d)\n        for d, j in enumerate([3, 6, 9, 12, 15], 1):\n            joint_to_chain[j], joint_to_depth[j] = (2, d)\n        for d, j in enumerate([14, 17, 19, 21], 4):\n            joint_to_chain[j], joint_to_depth[j] = (3, d)\n        for d, j in enumerate([13, 16, 18, 20], 4):\n            joint_to_chain[j], joint_to_depth[j] = (4, d)\n        self.register_buffer('joint_to_chain', torch.tensor(joint_to_chain))\n        self.register_buffer('joint_to_depth', torch.tensor(joint_to_depth))\n        self.chain_emb = nn.Embedding(5, model_dim // 2)\n        self.depth_emb = nn.Embedding(8, model_dim // 2)\n\n    def forward(self, joint_ids: torch.Tensor) -> torch.Tensor:\n        chains = self.joint_to_chain[joint_ids]\n        depths = self.joint_to_depth[joint_ids]\n        return torch.cat([self.chain_emb(chains), self.depth_emb(depths)], dim=-1)\n\nclass FlowMatchingPredictor(nn.Module):\n\n    def __init__(self, per_joint_dim: int=32, model_dim: int=64, num_layers: int=2, joint_count: int=22, time_embed_dim: int=64, dropout: float=0.1):\n        super().__init__()\n        self.model_dim = model_dim\n        self.num_layers = num_layers\n        self.joint_count = joint_count\n        self.time_embed_dim = time_embed_dim\n        self.kinematic_encoder = KinematicChainEncoder(model_dim)\n        self.time_mlp = nn.Sequential(nn.Linear(time_embed_dim, model_dim), nn.SiLU(), nn.Linear(model_dim, model_dim))\n        self.input_proj_history = nn.Linear(per_joint_dim, model_dim)\n        self.input_proj_prev_root = nn.Linear(9, model_dim)\n        self.input_proj_prev_joint = nn.Linear(12, model_dim)\n        self.input_proj_noisy_root = nn.Linear(9, model_dim)\n        self.input_proj_noisy_joint = nn.Linear(3, model_dim)\n        encoder_layer = nn.TransformerEncoderLayer(d_model=model_dim, nhead=4, dim_feedforward=model_dim * 2, dropout=dropout, activation='gelu', batch_first=True, norm_first=True)\n        self.spatial_transformer = nn.TransformerEncoder(encoder_layer, num_layers, enable_nested_tensor=False)\n        self.root_head = nn.Sequential(nn.Linear(model_dim, model_dim), nn.GELU(), nn.Linear(model_dim, 9))\n        self.joint_head = nn.Sequential(nn.Linear(model_dim, model_dim), nn.GELU(), nn.Linear(model_dim, 3))\n        self.null_prev_root = nn.Parameter(torch.zeros(1, 9))\n        self.null_prev_joint = nn.Parameter(torch.zeros(1, joint_count - 1, 12))\n\n    def _sinusoidal_time_embedding(self, t: torch.Tensor, max_positions=10000):\n        half_dim = self.time_embed_dim // 2\n        freqs = torch.exp(-torch.log(torch.tensor(max_positions)) / (half_dim - 1) * torch.arange(half_dim, device=t.device))\n        args = t.unsqueeze(-1) * freqs.unsqueeze(0)\n        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)\n        if self.time_embed_dim % 2 != 0:\n            emb = F.pad(emb, (0, 1))\n        return emb\n\n    def forward(self, history_features: torch.Tensor, noise_level: torch.Tensor, noisy_target: Optional[torch.Tensor]=None, prev_frame_features: Optional[torch.Tensor]=None, temporal_progress: Optional[torch.Tensor]=None):\n        B, J, _ = history_features.shape\n        device = history_features.device\n        if prev_frame_features is None:\n            prev_root = self.null_prev_root.expand(B, 9)\n            prev_joints = self.null_prev_joint.expand(B, J - 1, 12)\n        else:\n            prev_root = prev_frame_features[:, :9]\n            prev_joints = prev_frame_features[:, 9:]\n            prev_joints = prev_joints.reshape(B, J - 1, 12)\n        history_proj = self.input_proj_history(history_features)\n        prev_root_proj = self.input_proj_prev_root(prev_root).unsqueeze(1)\n        prev_joint_proj = self.input_proj_prev_joint(prev_joints)\n        prev_proj = torch.cat([prev_root_proj, prev_joint_proj], dim=1)\n        cond_proj = history_proj + prev_proj\n        if noisy_target is None:\n            noisy_target = torch.randn(B, 72, device=device)\n        noisy_root_in = noisy_target[:, :9]\n        noisy_joints_in = noisy_target[:, 9:].reshape(B, J - 1, 3)\n        noisy_root_proj = self.input_proj_noisy_root(noisy_root_in).unsqueeze(1)\n        noisy_joint_proj = self.input_proj_noisy_joint(noisy_joints_in)\n        noisy_proj = torch.cat([noisy_root_proj, noisy_joint_proj], dim=1)\n        x = cond_proj + noisy_proj\n        t_emb = self._sinusoidal_time_embedding(noise_level)\n        t_bias = self.time_mlp(t_emb)\n        x = x + t_bias.unsqueeze(1)\n        joint_ids = torch.arange(J, device=device)\n        kinematic_bias = self.kinematic_encoder(joint_ids)\n        x = x + kinematic_bias.unsqueeze(0)\n        x = self.spatial_transformer(x)\n        root_token = x[:, 0:1, :]\n        joint_tokens = x[:, 1:, :]\n        root_out = self.root_head(root_token)\n        joint_out = self.joint_head(joint_tokens)\n        pred_frame = torch.cat([root_out.squeeze(1), joint_out.reshape(B, -1)], dim=-1)\n        return pred_frame\n\nclass HumanMotionGenerator(nn.Module):\n\n    def __init__(self, encoder: MotionHistoryEncoder, predictor: FlowMatchingPredictor) -> None:\n        super().__init__()\n        self.encoder = encoder\n        self.predictor = predictor\n\n    def generate_sequence(self, text: Union[str, List[str], torch.Tensor], num_frames: int=200, num_steps: int=10, guidance_scale: float=2.5, input_features: Optional[torch.Tensor]=None, total_duration: Optional[torch.Tensor]=None, dataset_type: str='t2m') -> torch.Tensor:\n        from utils.train_utils import extract_prev_frame_features\n        self.eval()\n        with torch.no_grad():\n            if isinstance(text, str):\n                from utils.text_encoder import CLIPEncoder\n                clip_encoder = CLIPEncoder()\n                text = clip_encoder(text)\n                B = 1\n            elif isinstance(text, list):\n                from utils.text_encoder import CLIPEncoder\n                clip_encoder = CLIPEncoder()\n                text = clip_encoder(text)\n                B = text.shape[0]\n            else:\n                B = text.shape[0]\n            device = next(self.parameters()).device\n            if input_features is None:\n                null_features = self.encoder.null_history.expand(B, 1, -1).clone()\n                feature_history = null_features\n                null_positions = features_to_positions(null_features.squeeze(1), dataset_type=dataset_type)\n                position_history = null_positions.unsqueeze(1)\n            elif input_features.ndim == 2:\n                feature_history = input_features.unsqueeze(1).clone()\n                positions = features_to_positions(input_features, dataset_type=dataset_type)\n                position_history = positions.unsqueeze(1)\n            else:\n                feature_history = input_features.clone()\n                position_history = features_to_positions(input_features, dataset_type=dataset_type)\n            joint_sequence = []\n            for frame_idx in range(num_frames):\n                last_frame = feature_history[:, -1]\n                prev_positions = position_history[:, -1]\n                prev_root_pos = prev_positions[:, 0]\n                prev_root_rot_6d = last_frame[:, 69:75]\n                context_cond = self.encoder(batch_size=B, text=text, input_features=feature_history)[:, -1, :, :]\n                context_uncond = self.encoder(batch_size=B, text=None, input_features=feature_history)[:, -1, :, :]\n                prev_frame_features = extract_prev_frame_features(last_frame)\n                x_t = torch.randn((B, 72), device=device)\n                dt = 1.0 / num_steps\n                for step in range(num_steps):\n                    t = torch.full((B,), step * dt, device=device)\n                    v_cond = self.predictor(history_features=context_cond, noise_level=t, noisy_target=x_t, prev_frame_features=prev_frame_features)\n                    v_uncond = self.predictor(history_features=context_uncond, noise_level=t, noisy_target=x_t, prev_frame_features=prev_frame_features)\n                    v_t = v_uncond + guidance_scale * (v_cond - v_uncond)\n                    x_t = x_t + v_t * dt\n                new_positions = flow_output_to_positions(x_t, prev_root_pos, prev_root_rot_6d)\n                position_history = torch.cat([position_history, new_positions.unsqueeze(1)], dim=1)\n                feature_history = preprocess_sequence(position_history, dataset_type=dataset_type)\n                joint_sequence.append(new_positions)\n                if (frame_idx + 1) % 50 == 0:\n                    print(f'Generated {frame_idx + 1}/{num_frames} frames')\n            joint_positions = torch.stack(joint_sequence, dim=1)\n            return joint_positions\n\n    @classmethod\n    def load_from_checkpoint(cls, checkpoint_path: Union[str, Path], config: Config, device: str='cpu') -> 'HumanMotionGenerator':\n        print(f'Loading checkpoint from {checkpoint_path}...')\n        checkpoint = torch.load(checkpoint_path, map_location=device)\n        encoder = MotionHistoryEncoder(frame_feature_dim=config.motion_dim, text_embedding_dim=config.text_embedding_dim, per_joint_out_dim=config.per_joint_out_dim, joint_count=config.num_joints, model_dim=config.model_dim, num_layers=4, max_text_seq_len=config.max_text_seq_len, dropout=config.dropout)\n        predictor = FlowMatchingPredictor(per_joint_dim=config.per_joint_out_dim, model_dim=config.model_dim, num_layers=config.num_flow_layers, joint_count=config.num_joints)\n        if 'encoder_ema' in checkpoint and 'predictor_ema' in checkpoint:\n            print('Loading EMA weights for generation...')\n            encoder.load_state_dict(checkpoint['encoder_ema'])\n            predictor.load_state_dict(checkpoint['predictor_ema'])\n        else:\n            print('Loading standard weights (EMA not found)...')\n            encoder.load_state_dict(checkpoint['encoder'])\n            predictor.load_state_dict(checkpoint['predictor'])\n        encoder.to(device)\n        predictor.to(device)\n        encoder.eval()\n        predictor.eval()\n        return cls(encoder, predictor)",
    "requirements.txt": "# Core ML dependencies\ntorch\ntorchvision\nnumpy\nscipy\ntransformers\n\n# Data processing\npandas\n\n# Visualization\nmatplotlib\nseaborn\n\n# Utilities\ntqdm\ngdown\n\n# Text Encoding\nftfy\nregex\n",
    "utils/__init__.py": "# Utils module for motion generation project\n",
    "utils/utils.py": "from utils.dataset import Text2MotionDataset, create_dataloader, load_sample\nfrom utils.motion_utils import DATASET_CONFIGS, get_dataset_config, features_to_positions, preprocess_sequence, IncrementalFeatureExtractor\nfrom utils.visualization import plot_3d_motion, visualize_motion, compare_motions\n__all__ = ['Text2MotionDataset', 'create_dataloader', 'load_sample', 'DATASET_CONFIGS', 'get_dataset_config', 'features_to_positions', 'preprocess_sequence', 'IncrementalFeatureExtractor', 'plot_3d_motion', 'visualize_motion', 'compare_motions']",
    "utils/dataset.py": "import torch\nimport numpy as np\nfrom os.path import join as pjoin\nimport random\nfrom tqdm import tqdm\nfrom torch.utils.data import Dataset, DataLoader\nfrom pathlib import Path\nfrom typing import List, Dict, Any, Optional, Tuple\nfrom config import Config\n\nclass Text2MotionDataset(Dataset):\n\n    def __init__(self, config: Config, mean: np.ndarray, std: np.ndarray, split: str='train', feature_dims: tuple[slice, ...] | None=None):\n        self.config = config\n        self.feature_dims = feature_dims if feature_dims is not None else config.feature_dims\n        self.max_length = 20\n        self.pointer = 0\n        self.max_motion_length = config.max_motion_length\n        min_motion_len = 40 if config.dataset_name == 't2m' else 24\n        motion_dir = config.dataset_path / 'new_joint_vecs'\n        joints_dir = config.dataset_path / 'new_joints'\n        text_dir = config.dataset_path / 'texts'\n        split_file = config.dataset_path / f'{split}.txt'\n        data_dict = {}\n        id_list = []\n        with open(str(split_file), 'r', encoding='utf-8') as f:\n            for line in f.readlines():\n                id_list.append(line.strip())\n        new_name_list = []\n        length_list = []\n        for name in tqdm(id_list):\n            try:\n                motion = np.load(pjoin(str(motion_dir), name + '.npy'))\n                joints = np.load(pjoin(str(joints_dir), name + '.npy'))\n                if len(motion) < min_motion_len or len(motion) >= 200:\n                    continue\n                text_data = []\n                flag = False\n                with open(pjoin(str(text_dir), name + '.txt'), 'r', encoding='utf-8') as f:\n                    for line in f.readlines():\n                        text_dict: Dict[str, Optional[Any]] = {}\n                        line_split = line.strip().split('#')\n                        caption = line_split[0]\n                        tokens = line_split[1].split(' ')\n                        f_tag = float(line_split[2])\n                        to_tag = float(line_split[3])\n                        f_tag = 0.0 if np.isnan(f_tag) else f_tag\n                        to_tag = 0.0 if np.isnan(to_tag) else to_tag\n                        text_dict['caption'] = caption\n                        text_dict['tokens'] = tokens\n                        if f_tag == 0.0 and to_tag == 0.0:\n                            flag = True\n                            text_data.append(text_dict)\n                        else:\n                            try:\n                                n_motion = motion[int(f_tag * 20):int(to_tag * 20)]\n                                if len(n_motion) < min_motion_len or len(n_motion) >= 200:\n                                    continue\n                                new_name = random.choice('ABCDEFGHIJKLMNOPQRSTUVW') + '_' + name\n                                while new_name in data_dict:\n                                    new_name = random.choice('ABCDEFGHIJKLMNOPQRSTUVW') + '_' + name\n                                n_joints = joints[int(f_tag * 20):int(to_tag * 20)]\n                                data_dict[new_name] = {'motion': n_motion, 'joints': n_joints, 'length': len(n_motion), 'text': [text_dict]}\n                                new_name_list.append(new_name)\n                                length_list.append(len(n_motion))\n                            except:\n                                print(line_split)\n                                print(line_split[2], line_split[3], f_tag, to_tag, name)\n                if flag:\n                    data_dict[name] = {'motion': motion, 'joints': joints, 'length': len(motion), 'text': text_data}\n                    new_name_list.append(name)\n                    length_list.append(len(motion))\n            except Exception as e:\n                pass\n        name_list, length_list = (new_name_list, length_list)\n        self.mean = torch.from_numpy(mean).float()\n        self.std = torch.from_numpy(std).float()\n        self.length_arr = np.array(length_list)\n        self.data_dict = data_dict\n        self.name_list = name_list\n        self.text_cache_path = config.dataset_path / 'text_embeddings_cache.pt'\n        self.text_cache: Dict[str, torch.Tensor] = {}\n        if self.text_cache_path.exists():\n            print(f'Loading text embedding cache from {self.text_cache_path}...')\n            self.text_cache = torch.load(self.text_cache_path)\n        all_captions = set()\n        for key, data in self.data_dict.items():\n            for text_item in data['text']:\n                all_captions.add(text_item['caption'])\n        missing_captions = [cap for cap in all_captions if cap not in self.text_cache]\n        if missing_captions:\n            print(f'Computed {len(self.text_cache)}/{len(all_captions)} embeddings. Computing {len(missing_captions)} missing...')\n            from utils.text_encoder import CLIPEncoder\n            clip_encoder = CLIPEncoder(model_name='openai/clip-vit-base-patch32')\n            clip_encoder.to(config.device)\n            batch_size = 32\n            for i in tqdm(range(0, len(missing_captions), batch_size), desc='Encoding Texts'):\n                batch_caps = missing_captions[i:i + batch_size]\n                with torch.no_grad():\n                    embeddings = clip_encoder(batch_caps).cpu()\n                for cap, emb in zip(batch_caps, embeddings):\n                    self.text_cache[cap] = emb\n            print(f'Saving updated cache to {self.text_cache_path}...')\n            torch.save(self.text_cache, self.text_cache_path)\n            del clip_encoder\n            torch.cuda.empty_cache()\n        else:\n            print('All text embeddings are cached.')\n\n    def inv_transform(self, data):\n        return data * self.std + self.mean\n\n    def __len__(self):\n        return len(self.data_dict) - self.pointer\n    '\\n    FINAL CORRECT __getitem__ implementation\\n    This is the ONLY version that works - replace everything else\\n    '\n\n    def __getitem__(self, item):\n        idx = self.pointer + item\n        data = self.data_dict[self.name_list[idx]]\n        motion = data['motion']\n        joints = data['joints']\n        original_length = data['length']\n        text_list = data['text']\n        text_data = random.choice(text_list)\n        caption = text_data['caption']\n        motion = torch.from_numpy(motion.copy()).float()\n        joints = torch.from_numpy(joints.copy()).float()\n        motion = (motion - self.mean) / self.std\n        m_length = original_length\n        if self.config.unit_length < 10:\n            coin2 = np.random.choice(['single', 'single', 'double'])\n        else:\n            coin2 = 'single'\n        if coin2 == 'double':\n            m_length = (m_length // self.config.unit_length - 1) * self.config.unit_length\n        else:\n            m_length = m_length // self.config.unit_length * self.config.unit_length\n        m_length = min(m_length, len(motion))\n        m_length = max(1, m_length)\n        motion = motion[:m_length]\n        joints = joints[:m_length]\n        target_len = self.max_motion_length\n        current_len = len(motion)\n        if current_len < target_len:\n            pad_size = target_len - current_len\n            motion = torch.cat([motion, torch.zeros(pad_size, motion.shape[1], dtype=motion.dtype, device=motion.device)], dim=0)\n            joints = torch.cat([joints, torch.zeros(pad_size, joints.shape[1], joints.shape[2], dtype=joints.dtype, device=joints.device)], dim=0)\n        elif current_len > target_len:\n            motion = motion[:target_len]\n            joints = joints[:target_len]\n        assert motion.shape[0] == target_len, f'Motion shape[0]={motion.shape[0]}, expected {target_len}'\n        assert motion.shape[1] == 271, f'Motion shape[1]={motion.shape[1]}, expected 271'\n        assert joints.shape[0] == target_len, f'Joints shape[0]={joints.shape[0]}, expected {target_len}'\n        text_embedding = self.text_cache[caption]\n        if isinstance(text_embedding, np.ndarray):\n            text_embedding = torch.from_numpy(text_embedding).float()\n        else:\n            text_embedding = text_embedding.float()\n        return (caption, motion, joints, m_length, text_embedding)\n\n    def reset_min_len(self, length):\n        assert length <= self.max_motion_length\n        self.pointer = np.searchsorted(self.length_arr, length)\n        print('Pointer Pointing at %d' % self.pointer)\nfrom typing import List, Dict, Any\nCLIP_MAX_SEQ_LEN = 77\nCLIP_EMBED_DIM = 512\n\ndef text2motion_collate_fn(batch: List[Tuple[str, torch.Tensor, torch.Tensor, int, torch.Tensor]]) -> Dict[str, Any]:\n    captions = [b[0] for b in batch]\n    motions_list = [b[1] for b in batch]\n    joints_list = [b[2] for b in batch]\n    lengths = [b[3] for b in batch]\n    text_embs_list = [b[4] for b in batch]\n\n    def to_tensor(x):\n        if isinstance(x, np.ndarray):\n            return torch.from_numpy(x).float()\n        elif isinstance(x, torch.Tensor):\n            return x.float()\n        else:\n            return torch.tensor(x).float()\n    motions_list = [to_tensor(x) for x in motions_list]\n    joints_list = [to_tensor(x) for x in joints_list]\n    text_embs_list = [to_tensor(x) for x in text_embs_list]\n    motion_batch = torch.stack(motions_list, dim=0)\n    joints_batch = torch.stack(joints_list, dim=0)\n    length_batch = torch.tensor(lengths, dtype=torch.long)\n    text_emb_batch = torch.stack(text_embs_list, dim=0)\n    return {'captions': captions, 'motion': motion_batch, 'joints': joints_batch, 'lengths': length_batch, 'text_clip': text_emb_batch}\n\ndef create_dataloader(config: Config, split: str='train', shuffle: bool=True) -> DataLoader:\n    mean_path = config.dataset_path / 'Mean.npy'\n    std_path = config.dataset_path / 'Std.npy'\n    if not mean_path.exists() or not std_path.exists():\n        raise FileNotFoundError(f'Mean.npy and/or Std.npy not found in {config.dataset_path}. Please ensure Mean.npy and Std.npy exist in the dataset directory.')\n    mean = np.load(mean_path)\n    std = np.load(std_path)\n    dataset_obj = Text2MotionDataset(config, mean, std, split, feature_dims=config.feature_dims)\n    return DataLoader(dataset_obj, batch_size=config.batch_size, shuffle=shuffle, num_workers=config.num_workers, pin_memory=config.pin_memory, collate_fn=text2motion_collate_fn)\n\ndef load_sample(dataset_path: Path, file_id: str) -> Dict[str, Optional[Any]]:\n    features_path = dataset_path / 'new_joint_vecs' / f'{file_id}.npy'\n    joints_path = dataset_path / 'new_joints' / f'{file_id}.npy'\n    text_path = dataset_path / 'texts' / f'{file_id}.txt'\n    data: Dict[str, Optional[Any]] = {'file_id': file_id}\n    if features_path.exists():\n        data['features'] = np.load(features_path)\n    else:\n        print(f'Warning: Features not found for {file_id}')\n        data['features'] = None\n    if joints_path.exists():\n        data['joints'] = np.load(joints_path)\n    else:\n        print(f'Warning: Joints not found for {file_id}')\n        data['joints'] = None\n    if text_path.exists():\n        with open(text_path, 'r') as f:\n            descriptions = [line.strip().split('#')[0] for line in f.readlines()]\n            data['text'] = descriptions[0] if descriptions else ''\n    else:\n        data['text'] = ''\n    return data",
    "utils/motion_utils.py": "import torch\nimport numpy as np\nfrom typing import List, Tuple, Dict, Any, Optional\nfrom utils.quaternion import qrot, qinv, qmul, quaternion_to_cont6d, cont6d_to_matrix, cont6d_to_quaternion\nT2M_RAW_OFFSETS = torch.tensor([[0, 0, 0], [1, 0, 0], [-1, 0, 0], [0, 1, 0], [0, -1, 0], [0, -1, 0], [0, 1, 0], [0, -1, 0], [0, -1, 0], [0, 1, 0], [0, 0, 1], [0, 0, 1], [0, 1, 0], [1, 0, 0], [-1, 0, 0], [0, 0, 1], [0, -1, 0], [0, -1, 0], [0, -1, 0], [0, -1, 0], [0, -1, 0], [0, -1, 0]], dtype=torch.float32)\nT2M_KINEMATIC_CHAIN = [[0, 2, 5, 8, 11], [0, 1, 4, 7, 10], [0, 3, 6, 9, 12, 15], [9, 14, 17, 19, 21], [9, 13, 16, 18, 20]]\nDATASET_CONFIGS = {'t2m': {'name': 'HumanML3D', 'num_joints': 22, 'feature_dim': 271, 'raw_offsets': T2M_RAW_OFFSETS, 'kinematic_chain': T2M_KINEMATIC_CHAIN, 'face_joint_indx': [2, 1, 17, 16], 'fid_r': [8, 11], 'fid_l': [7, 10]}}\n\ndef get_dataset_config(dataset_type: str='t2m') -> Dict[str, Any]:\n    if dataset_type not in DATASET_CONFIGS:\n        raise ValueError(f'Unknown dataset_type: {dataset_type}. Available: {list(DATASET_CONFIGS.keys())}')\n    return DATASET_CONFIGS[dataset_type]\nFEATURE_SLICES = {'root_features': slice(0, 3), 'ric_positions': slice(3, 69), 'rotations_6d': slice(69, 201), 'local_velocities': slice(201, 267), 'foot_contacts': slice(267, 271)}\n\ndef _compute_ik(positions: torch.Tensor, raw_offsets: torch.Tensor, kinematic_chain: List[List[int]], face_joint_indx: List[int]) -> torch.Tensor:\n    batch_shape = positions.shape[:-2]\n    device = positions.device\n    dtype = positions.dtype\n    positions_flat = positions.reshape(-1, 22, 3)\n    B = positions_flat.shape[0]\n    l_hip, r_hip, sdr_r, sdr_l = face_joint_indx\n    across1 = positions_flat[:, r_hip] - positions_flat[:, l_hip]\n    across2 = positions_flat[:, sdr_r] - positions_flat[:, sdr_l]\n    across = across1 + across2\n    across = across / (torch.norm(across, dim=-1, keepdim=True) + 1e-10)\n    forward = torch.cross(torch.tensor([[0, 1, 0]], device=device, dtype=dtype).expand(B, -1), across, dim=-1)\n    forward = forward / (torch.norm(forward, dim=-1, keepdim=True) + 1e-10)\n    target = torch.tensor([[0, 0, 1]], device=device, dtype=dtype).expand(B, -1)\n    root_quat = _qbetween(forward, target)\n    quaternions = torch.zeros(B, 22, 4, device=device, dtype=dtype)\n    quaternions[:, 0] = root_quat\n    offsets = raw_offsets.unsqueeze(0).expand(B, -1, -1)\n    for chain in kinematic_chain:\n        R = root_quat\n        for i in range(len(chain) - 1):\n            parent_idx = chain[i]\n            child_idx = chain[i + 1]\n            u = offsets[:, child_idx]\n            v = positions_flat[:, child_idx] - positions_flat[:, parent_idx]\n            v = v / (torch.norm(v, dim=-1, keepdim=True) + 1e-10)\n            rot_u_v = _qbetween(u, v)\n            R_loc = qmul(qinv(R), rot_u_v)\n            quaternions[:, child_idx] = R_loc\n            R = qmul(R, R_loc)\n    return quaternions.reshape(batch_shape + (22, 4))\n\ndef _qbetween(v0: torch.Tensor, v1: torch.Tensor) -> torch.Tensor:\n    v0 = v0 / (torch.norm(v0, dim=-1, keepdim=True) + 1e-10)\n    v1 = v1 / (torch.norm(v1, dim=-1, keepdim=True) + 1e-10)\n    dot = (v0 * v1).sum(dim=-1, keepdim=True)\n    cross = torch.cross(v0, v1, dim=-1)\n    w = 1.0 + dot\n    q = torch.cat([w, cross], dim=-1)\n    q = q / (torch.norm(q, dim=-1, keepdim=True) + 1e-10)\n    return q\n\ndef _forward_kinematics(rotations_6d: torch.Tensor, root_pos: torch.Tensor, offsets: torch.Tensor, kinematic_chain: List[List[int]]) -> torch.Tensor:\n    batch_shape = rotations_6d.shape[:-2]\n    device = rotations_6d.device\n    dtype = rotations_6d.dtype\n    rotations_flat = rotations_6d.reshape(-1, 22, 6)\n    root_pos_flat = root_pos.reshape(-1, 3)\n    B = rotations_flat.shape[0]\n    positions = torch.zeros(B, 22, 3, device=device, dtype=dtype)\n    positions[:, 0] = root_pos_flat\n    offsets_expanded = offsets.unsqueeze(0).expand(B, -1, -1)\n    for chain in kinematic_chain:\n        matR = cont6d_to_matrix(rotations_flat[:, 0])\n        for i in range(1, len(chain)):\n            child_idx = chain[i]\n            parent_idx = chain[i - 1]\n            child_rot = cont6d_to_matrix(rotations_flat[:, child_idx])\n            matR = torch.bmm(matR, child_rot)\n            offset_vec = offsets_expanded[:, child_idx].unsqueeze(-1)\n            positions[:, child_idx] = torch.bmm(matR, offset_vec).squeeze(-1) + positions[:, parent_idx]\n    return positions.reshape(batch_shape + (22, 3))\n\ndef preprocess_sequence(positions: torch.Tensor, dataset_type: str='t2m', feet_thre: float=0.002) -> torch.Tensor:\n    config = get_dataset_config(dataset_type)\n    raw_offsets = config['raw_offsets']\n    kinematic_chain = config['kinematic_chain']\n    face_joint_indx = config['face_joint_indx']\n    fid_r = config['fid_r']\n    fid_l = config['fid_l']\n    device = positions.device\n    dtype = positions.dtype\n    if positions.ndim == 4:\n        B, N, J, _ = positions.shape\n        features_batch = []\n        for b in range(B):\n            pos_b = positions[b]\n            feat_b = preprocess_sequence(pos_b, dataset_type, feet_thre)\n            features_batch.append(feat_b)\n        return torch.stack(features_batch, dim=0)\n    N = positions.shape[0]\n    root_features = torch.zeros(N, 3, device=device, dtype=dtype)\n    root_features[:, 0] = positions[:, 0, 1]\n    if N > 1:\n        root_features[1:, 1] = positions[1:, 0, 0] - positions[:-1, 0, 0]\n        root_features[1:, 2] = positions[1:, 0, 2] - positions[:-1, 0, 2]\n    quaternions = _compute_ik(positions, raw_offsets, kinematic_chain, face_joint_indx)\n    root_quat = quaternions[:, 0].clone()\n    ric = positions - positions[:, 0:1, :]\n    ric = qrot(root_quat.unsqueeze(1).expand(-1, 22, -1), ric)\n    rotations_6d = quaternion_to_cont6d(quaternions)\n    local_vel = torch.zeros(N, 22, 3, device=device, dtype=dtype)\n    if N > 1:\n        local_vel[1:] = qrot(root_quat[1:].unsqueeze(1).expand(-1, 22, -1), positions[1:] - positions[:-1])\n    feet_l = torch.zeros(N, 2, device=device, dtype=dtype)\n    feet_r = torch.zeros(N, 2, device=device, dtype=dtype)\n    if N > 1:\n        vel_l = positions[1:, fid_l] - positions[:-1, fid_l]\n        vel_r = positions[1:, fid_r] - positions[:-1, fid_r]\n        feet_l[1:] = (torch.sum(vel_l ** 2, dim=-1) < feet_thre).float()\n        feet_r[1:] = (torch.sum(vel_r ** 2, dim=-1) < feet_thre).float()\n    features = torch.cat([root_features, ric.reshape(N, -1), rotations_6d.reshape(N, -1), local_vel.reshape(N, -1), feet_l, feet_r], dim=-1)\n    return features\n\ndef features_to_positions(features: torch.Tensor, dataset_type: str='t2m') -> torch.Tensor:\n    root_features = features[..., 0:3]\n    ric = features[..., 3:69].reshape(features.shape[:-1] + (22, 3))\n    rotations_6d = features[..., 69:201].reshape(features.shape[:-1] + (22, 6))\n    root_quat = cont6d_to_quaternion(rotations_6d[..., 0, :])\n    root_height_y = root_features[..., 0:1]\n    root_vel_x = root_features[..., 1:2]\n    root_vel_z = root_features[..., 2:3]\n    if features.ndim == 2:\n        root_pos_x = torch.cumsum(root_vel_x, dim=0)\n        root_pos_z = torch.cumsum(root_vel_z, dim=0)\n    else:\n        root_pos_x = torch.cumsum(root_vel_x, dim=-2)\n        root_pos_z = torch.cumsum(root_vel_z, dim=-2)\n    global_root_pos = torch.cat([root_pos_x, root_height_y, root_pos_z], dim=-1)\n    root_quat_expanded = root_quat.unsqueeze(-2).expand(root_quat.shape[:-1] + (22, -1))\n    positions = global_root_pos.unsqueeze(-2) + qrot(qinv(root_quat_expanded), ric)\n    return positions\n\ndef flow_output_to_positions(flow_output: torch.Tensor, prev_root_pos: torch.Tensor, prev_root_rot_6d: torch.Tensor) -> torch.Tensor:\n    B = flow_output.shape[0]\n    device = flow_output.device\n    dtype = flow_output.dtype\n    root_height = flow_output[:, 0:1]\n    root_vel = flow_output[:, 1:3]\n    root_rot_6d = flow_output[:, 3:9]\n    joint_ric = flow_output[:, 9:72].reshape(B, 21, 3)\n    new_root_x = prev_root_pos[:, 0:1] + root_vel[:, 0:1]\n    new_root_y = root_height\n    new_root_z = prev_root_pos[:, 2:3] + root_vel[:, 1:2]\n    new_root_pos = torch.cat([new_root_x, new_root_y, new_root_z], dim=-1)\n    root_quat = cont6d_to_quaternion(root_rot_6d)\n    root_quat_expanded = root_quat.unsqueeze(1).expand(-1, 21, -1)\n    global_joint_offsets = qrot(qinv(root_quat_expanded), joint_ric)\n    global_joints = new_root_pos.unsqueeze(1) + global_joint_offsets\n    positions = torch.cat([new_root_pos.unsqueeze(1), global_joints], dim=1)\n    return positions\n\ndef flow_output_to_displacements(flow_output: torch.Tensor) -> torch.Tensor:\n    B = flow_output.shape[0]\n    device = flow_output.device\n    dtype = flow_output.dtype\n    root_disp_x = flow_output[:, 1:2]\n    root_disp_z = flow_output[:, 2:3]\n    root_disp_y = torch.zeros_like(root_disp_x)\n    root_disp = torch.cat([root_disp_x, root_disp_y, root_disp_z], dim=-1)\n    joint_disps = flow_output[:, 9:72].reshape(B, 21, 3)\n    displacements = torch.cat([root_disp.unsqueeze(1), joint_disps], dim=1)\n    return displacements\n\nclass IncrementalFeatureExtractor:\n\n    def __init__(self, dataset_type: str='t2m', feet_thre: float=0.002, device: torch.device=torch.device('cpu'), dtype: torch.dtype=torch.float32):\n        config = get_dataset_config(dataset_type)\n        self.raw_offsets = config['raw_offsets'].to(device).to(dtype)\n        self.kinematic_chain = config['kinematic_chain']\n        self.face_joint_indx = config['face_joint_indx']\n        self.fid_r = config['fid_r']\n        self.fid_l = config['fid_l']\n        self.feet_thre = feet_thre\n        self.device = device\n        self.dtype = dtype\n        self.prev_positions: Optional[torch.Tensor] = None\n        self.is_initialized = False\n\n    def initialize(self, initial_positions: torch.Tensor) -> torch.Tensor:\n        initial_positions = initial_positions.to(self.device).to(self.dtype)\n        B = initial_positions.shape[0]\n        self.prev_positions = initial_positions.clone()\n        quaternions = _compute_ik(initial_positions, self.raw_offsets, self.kinematic_chain, self.face_joint_indx)\n        rotations_6d = quaternion_to_cont6d(quaternions)\n        root_pos = initial_positions[:, 0]\n        fk_positions = _forward_kinematics(rotations_6d, root_pos, self.raw_offsets, self.kinematic_chain)\n        self.prev_fk_positions = fk_positions\n        self.is_initialized = True\n        return torch.zeros(B, 271, device=self.device, dtype=self.dtype)\n\n    def process_frame(self, positions: torch.Tensor) -> torch.Tensor:\n        positions = positions.to(self.device).to(self.dtype)\n        if not self.is_initialized:\n            return self.initialize(positions)\n        B = positions.shape[0]\n        root_height_y = positions[:, 0, 1:2]\n        root_vel_x = positions[:, 0, 0:1] - self.prev_positions[:, 0, 0:1]\n        root_vel_z = positions[:, 0, 2:3] - self.prev_positions[:, 0, 2:3]\n        root_features = torch.cat([root_height_y, root_vel_x, root_vel_z], dim=-1)\n        quaternions = _compute_ik(positions, self.raw_offsets, self.kinematic_chain, self.face_joint_indx)\n        root_quat = quaternions[:, 0]\n        rotations_6d = quaternion_to_cont6d(quaternions)\n        global_root_pos = positions[:, 0]\n        fk_positions = _forward_kinematics(rotations_6d, global_root_pos, self.raw_offsets, self.kinematic_chain)\n        ric = fk_positions - fk_positions[:, 0:1]\n        ric = qrot(root_quat.unsqueeze(1).expand(-1, 22, -1), ric)\n        local_vel = qrot(root_quat.unsqueeze(1).expand(-1, 22, -1), positions - self.prev_positions)\n        foot_vel = positions - self.prev_positions\n        feet_l = (torch.sum(foot_vel[:, self.fid_l] ** 2, dim=-1) < self.feet_thre).float()\n        feet_r = (torch.sum(foot_vel[:, self.fid_r] ** 2, dim=-1) < self.feet_thre).float()\n        self.prev_positions = positions.clone()\n        features = torch.cat([root_features, ric.reshape(B, -1), rotations_6d.reshape(B, -1), local_vel.reshape(B, -1), feet_l, feet_r], dim=-1)\n        return features\n\n    def reset(self):\n        self.prev_positions = None\n        self.prev_fk_positions = None\n        self.is_initialized = False\n\n    def process_flow_output(self, flow_output: torch.Tensor, prev_frame_271d: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:\n        prev_root_height = prev_frame_271d[:, 0:1]\n        if self.prev_positions is not None:\n            prev_root_pos = self.prev_positions[:, 0].clone()\n        else:\n            prev_root_pos = torch.zeros(flow_output.shape[0], 3, device=flow_output.device, dtype=flow_output.dtype)\n            prev_root_pos[:, 1] = prev_root_height.squeeze(-1)\n        prev_root_rot_6d = prev_frame_271d[:, 69:75]\n        new_positions = flow_output_to_positions(flow_output, prev_root_pos, prev_root_rot_6d)\n        new_frame_271d = self.process_frame(new_positions)\n        return (new_frame_271d, new_positions)\n\ndef flow_output_to_271d(flow_output: torch.Tensor, prev_frame_271d: torch.Tensor, extractor: IncrementalFeatureExtractor) -> Tuple[torch.Tensor, torch.Tensor]:\n    return extractor.process_flow_output(flow_output, prev_frame_271d)",
    "utils/visualization.py": "import numpy as np\nimport matplotlib.pyplot as plt\nfrom matplotlib.animation import FuncAnimation\nfrom pathlib import Path\nfrom typing import Optional, Any\nfrom utils.motion_utils import T2M_KINEMATIC_CHAIN\n\ndef plot_3d_motion(motion: np.ndarray, fps: float=20, radius: float=1.0, title: str='Motion Visualization', follow_root: bool=False) -> FuncAnimation:\n    fig = plt.figure(figsize=(8, 8))\n    ax = fig.add_subplot(111, projection='3d')\n    ax.view_init(elev=15, azim=-70)\n    colors = ['#2980b9', '#c0392b', '#27ae60', '#f39c12', '#8e44ad']\n    lines = [ax.plot([], [], [], color=colors[i % len(colors)], marker='o', ms=2, lw=2)[0] for i in range(len(T2M_KINEMATIC_CHAIN))]\n    ax.set_xlabel('X (Side)')\n    ax.set_ylabel('Z (Forward)')\n    ax.set_zlabel('Y (Height)')\n    ax.set_title(title)\n    pos_min = motion.min(axis=(0, 1))\n    pos_max = motion.max(axis=(0, 1))\n\n    def update(frame):\n        root = motion[frame, 0, :]\n        if follow_root:\n            ax.set_xlim3d([root[0] - radius, root[0] + radius])\n            ax.set_ylim3d([root[2] - radius, root[2] + radius])\n            ax.set_zlim3d([pos_min[1], pos_max[1] + radius * 0.5])\n        else:\n            ax.set_xlim3d([pos_min[0] - radius, pos_max[0] + radius])\n            ax.set_ylim3d([pos_min[2] - radius, pos_max[2] + radius])\n            ax.set_zlim3d([pos_min[1], pos_max[1] + radius * 0.5])\n        for i, c_indices in enumerate(T2M_KINEMATIC_CHAIN):\n            joints = motion[frame, c_indices, :]\n            lines[i].set_data(joints[:, 0], joints[:, 2])\n            lines[i].set_3d_properties(joints[:, 1])\n        return lines\n    ani = FuncAnimation(fig, update, frames=len(motion), interval=1000 / fps, blit=False)\n    plt.close()\n    return ani\n\ndef visualize_motion(joint_positions: np.ndarray, ground_truth: Optional[np.ndarray]=None, title: str='Motion Visualization', save_path: Optional[Path]=None, fps: float=20, skip_frames: int=1, notebook: bool=True) -> Any:\n    fps = fps / skip_frames\n    ani = plot_3d_motion(joint_positions[::skip_frames], fps=fps, title=title)\n    if save_path:\n        save_path.parent.mkdir(parents=True, exist_ok=True)\n        ani.save(str(save_path), writer='ffmpeg', fps=int(fps))\n        print(f'Saved animation to {save_path}')\n    if notebook:\n        from IPython.display import HTML\n        return HTML(ani.to_html5_video())\n    return ani\n\ndef compare_motions(generated_joints: np.ndarray, ground_truth_joints: np.ndarray, save_path: Optional[Path]=None) -> None:\n    visualize_motion(generated_joints, ground_truth=ground_truth_joints, title='Generated vs Ground Truth', save_path=save_path)",
    "utils/quaternion.py": "import torch\nimport numpy as np\n_EPS4 = np.finfo(float).eps * 4.0\n_FLOAT_EPS = np.finfo(np.float64).eps\n\ndef qinv(q):\n    assert q.shape[-1] == 4, 'q must be a tensor of shape (*, 4)'\n    mask = torch.ones_like(q)\n    mask[..., 1:] = -mask[..., 1:]\n    return q * mask\n\ndef qinv_np(q):\n    assert q.shape[-1] == 4, 'q must be a tensor of shape (*, 4)'\n    return qinv(torch.from_numpy(q).float()).numpy()\n\ndef qnormalize(q):\n    assert q.shape[-1] == 4, 'q must be a tensor of shape (*, 4)'\n    return q / torch.norm(q, dim=-1, keepdim=True)\n\ndef qmul(q, r):\n    assert q.shape[-1] == 4\n    assert r.shape[-1] == 4\n    original_shape = q.shape\n    terms = torch.bmm(r.view(-1, 4, 1), q.view(-1, 1, 4))\n    w = terms[:, 0, 0] - terms[:, 1, 1] - terms[:, 2, 2] - terms[:, 3, 3]\n    x = terms[:, 0, 1] + terms[:, 1, 0] - terms[:, 2, 3] + terms[:, 3, 2]\n    y = terms[:, 0, 2] + terms[:, 1, 3] + terms[:, 2, 0] - terms[:, 3, 1]\n    z = terms[:, 0, 3] - terms[:, 1, 2] + terms[:, 2, 1] + terms[:, 3, 0]\n    return torch.stack((w, x, y, z), dim=1).view(original_shape)\n\ndef qrot(q, v):\n    assert q.shape[-1] == 4\n    assert v.shape[-1] == 3\n    assert q.shape[:-1] == v.shape[:-1]\n    original_shape = list(v.shape)\n    q = q.contiguous().view(-1, 4)\n    v = v.contiguous().view(-1, 3)\n    qvec = q[:, 1:]\n    uv = torch.cross(qvec, v, dim=1)\n    uuv = torch.cross(qvec, uv, dim=1)\n    return (v + 2 * (q[:, :1] * uv + uuv)).view(original_shape)\n\ndef qeuler(q, order, epsilon=0, deg=True):\n    assert q.shape[-1] == 4\n    original_shape = list(q.shape)\n    original_shape[-1] = 3\n    q = q.view(-1, 4)\n    q0 = q[:, 0]\n    q1 = q[:, 1]\n    q2 = q[:, 2]\n    q3 = q[:, 3]\n    if order == 'xyz':\n        x = torch.atan2(2 * (q0 * q1 - q2 * q3), 1 - 2 * (q1 * q1 + q2 * q2))\n        y = torch.asin(torch.clamp(2 * (q1 * q3 + q0 * q2), -1 + epsilon, 1 - epsilon))\n        z = torch.atan2(2 * (q0 * q3 - q1 * q2), 1 - 2 * (q2 * q2 + q3 * q3))\n    elif order == 'yzx':\n        x = torch.atan2(2 * (q0 * q1 - q2 * q3), 1 - 2 * (q1 * q1 + q3 * q3))\n        y = torch.atan2(2 * (q0 * q2 - q1 * q3), 1 - 2 * (q2 * q2 + q3 * q3))\n        z = torch.asin(torch.clamp(2 * (q1 * q2 + q0 * q3), -1 + epsilon, 1 - epsilon))\n    elif order == 'zxy':\n        x = torch.asin(torch.clamp(2 * (q0 * q1 + q2 * q3), -1 + epsilon, 1 - epsilon))\n        y = torch.atan2(2 * (q0 * q2 - q1 * q3), 1 - 2 * (q1 * q1 + q2 * q2))\n        z = torch.atan2(2 * (q0 * q3 - q1 * q2), 1 - 2 * (q1 * q1 + q3 * q3))\n    elif order == 'xzy':\n        x = torch.atan2(2 * (q0 * q1 + q2 * q3), 1 - 2 * (q1 * q1 + q3 * q3))\n        y = torch.atan2(2 * (q0 * q2 + q1 * q3), 1 - 2 * (q2 * q2 + q3 * q3))\n        z = torch.asin(torch.clamp(2 * (q0 * q3 - q1 * q2), -1 + epsilon, 1 - epsilon))\n    elif order == 'yxz':\n        x = torch.asin(torch.clamp(2 * (q0 * q1 - q2 * q3), -1 + epsilon, 1 - epsilon))\n        y = torch.atan2(2 * (q1 * q3 + q0 * q2), 1 - 2 * (q1 * q1 + q2 * q2))\n        z = torch.atan2(2 * (q1 * q2 + q0 * q3), 1 - 2 * (q1 * q1 + q3 * q3))\n    elif order == 'zyx':\n        x = torch.atan2(2 * (q0 * q1 + q2 * q3), 1 - 2 * (q1 * q1 + q2 * q2))\n        y = torch.asin(torch.clamp(2 * (q0 * q2 - q1 * q3), -1 + epsilon, 1 - epsilon))\n        z = torch.atan2(2 * (q0 * q3 + q1 * q2), 1 - 2 * (q2 * q2 + q3 * q3))\n    else:\n        raise\n    if deg:\n        return torch.stack((x, y, z), dim=1).view(original_shape) * 180 / np.pi\n    else:\n        return torch.stack((x, y, z), dim=1).view(original_shape)\n\ndef qmul_np(q, r):\n    q = torch.from_numpy(q).contiguous().float()\n    r = torch.from_numpy(r).contiguous().float()\n    return qmul(q, r).numpy()\n\ndef qrot_np(q, v):\n    q = torch.from_numpy(q).contiguous().float()\n    v = torch.from_numpy(v).contiguous().float()\n    return qrot(q, v).numpy()\n\ndef qeuler_np(q, order, epsilon=0, use_gpu=False):\n    if use_gpu:\n        q = torch.from_numpy(q).cuda().float()\n        return qeuler(q, order, epsilon).cpu().numpy()\n    else:\n        q = torch.from_numpy(q).contiguous().float()\n        return qeuler(q, order, epsilon).numpy()\n\ndef qfix(q):\n    assert len(q.shape) == 3\n    assert q.shape[-1] == 4\n    result = q.copy()\n    dot_products = np.sum(q[1:] * q[:-1], axis=2)\n    mask = dot_products < 0\n    mask = (np.cumsum(mask, axis=0) % 2).astype(bool)\n    result[1:][mask] *= -1\n    return result\n\ndef euler2quat(e, order, deg=True):\n    assert e.shape[-1] == 3\n    original_shape = list(e.shape)\n    original_shape[-1] = 4\n    e = e.view(-1, 3)\n    if deg:\n        e = e * np.pi / 180.0\n    x = e[:, 0]\n    y = e[:, 1]\n    z = e[:, 2]\n    rx = torch.stack((torch.cos(x / 2), torch.sin(x / 2), torch.zeros_like(x), torch.zeros_like(x)), dim=1)\n    ry = torch.stack((torch.cos(y / 2), torch.zeros_like(y), torch.sin(y / 2), torch.zeros_like(y)), dim=1)\n    rz = torch.stack((torch.cos(z / 2), torch.zeros_like(z), torch.zeros_like(z), torch.sin(z / 2)), dim=1)\n    result = None\n    for coord in order:\n        if coord == 'x':\n            r = rx\n        elif coord == 'y':\n            r = ry\n        elif coord == 'z':\n            r = rz\n        else:\n            raise\n        if result is None:\n            result = r\n        else:\n            result = qmul(result, r)\n    if order in ['xyz', 'yzx', 'zxy']:\n        result *= -1\n    return result.view(original_shape)\n\ndef expmap_to_quaternion(e):\n    assert e.shape[-1] == 3\n    original_shape = list(e.shape)\n    original_shape[-1] = 4\n    e = e.reshape(-1, 3)\n    theta = np.linalg.norm(e, axis=1).reshape(-1, 1)\n    w = np.cos(0.5 * theta).reshape(-1, 1)\n    xyz = 0.5 * np.sinc(0.5 * theta / np.pi) * e\n    return np.concatenate((w, xyz), axis=1).reshape(original_shape)\n\ndef euler_to_quaternion(e, order):\n    assert e.shape[-1] == 3\n    original_shape = list(e.shape)\n    original_shape[-1] = 4\n    e = e.reshape(-1, 3)\n    x = e[:, 0]\n    y = e[:, 1]\n    z = e[:, 2]\n    rx = np.stack((np.cos(x / 2), np.sin(x / 2), np.zeros_like(x), np.zeros_like(x)), axis=1)\n    ry = np.stack((np.cos(y / 2), np.zeros_like(y), np.sin(y / 2), np.zeros_like(y)), axis=1)\n    rz = np.stack((np.cos(z / 2), np.zeros_like(z), np.zeros_like(z), np.sin(z / 2)), axis=1)\n    result = None\n    for coord in order:\n        if coord == 'x':\n            r = rx\n        elif coord == 'y':\n            r = ry\n        elif coord == 'z':\n            r = rz\n        else:\n            raise\n        if result is None:\n            result = r\n        else:\n            result = qmul_np(result, r)\n    if order in ['xyz', 'yzx', 'zxy']:\n        result *= -1\n    return result.reshape(original_shape)\n\ndef quaternion_to_matrix(quaternions):\n    r, i, j, k = torch.unbind(quaternions, -1)\n    two_s = 2.0 / (quaternions * quaternions).sum(-1)\n    o = torch.stack((1 - two_s * (j * j + k * k), two_s * (i * j - k * r), two_s * (i * k + j * r), two_s * (i * j + k * r), 1 - two_s * (i * i + k * k), two_s * (j * k - i * r), two_s * (i * k - j * r), two_s * (j * k + i * r), 1 - two_s * (i * i + j * j)), -1)\n    return o.reshape(quaternions.shape[:-1] + (3, 3))\n\ndef quaternion_to_matrix_np(quaternions):\n    q = torch.from_numpy(quaternions).contiguous().float()\n    return quaternion_to_matrix(q).numpy()\n\ndef quaternion_to_cont6d_np(quaternions):\n    rotation_mat = quaternion_to_matrix_np(quaternions)\n    cont_6d = np.concatenate([rotation_mat[..., 0], rotation_mat[..., 1]], axis=-1)\n    return cont_6d\n\ndef quaternion_to_cont6d(quaternions):\n    rotation_mat = quaternion_to_matrix(quaternions)\n    cont_6d = torch.cat([rotation_mat[..., 0], rotation_mat[..., 1]], dim=-1)\n    return cont_6d\n\ndef cont6d_to_matrix(cont6d):\n    assert cont6d.shape[-1] == 6, 'The last dimension must be 6'\n    x_raw = cont6d[..., 0:3]\n    y_raw = cont6d[..., 3:6]\n    eps = 1e-08\n    x_norm = torch.norm(x_raw, dim=-1, keepdim=True).clamp(min=eps)\n    x = x_raw / x_norm\n    z = torch.cross(x, y_raw, dim=-1)\n    z_norm = torch.norm(z, dim=-1, keepdim=True).clamp(min=eps)\n    z = z / z_norm\n    y = torch.cross(z, x, dim=-1)\n    x = x[..., None]\n    y = y[..., None]\n    z = z[..., None]\n    mat = torch.cat([x, y, z], dim=-1)\n    return mat\n\ndef cont6d_to_matrix_np(cont6d):\n    q = torch.from_numpy(cont6d).contiguous().float()\n    return cont6d_to_matrix(q).numpy()\n\ndef matrix_to_quaternion(rotation_matrix):\n    batch_shape = rotation_matrix.shape[:-2]\n    rotation_matrix = rotation_matrix.reshape(-1, 3, 3)\n    batch_size = rotation_matrix.shape[0]\n    q = torch.zeros(batch_size, 4, device=rotation_matrix.device, dtype=rotation_matrix.dtype)\n    trace = rotation_matrix[:, 0, 0] + rotation_matrix[:, 1, 1] + rotation_matrix[:, 2, 2]\n    mask1 = trace > 0\n    s1 = torch.sqrt(trace[mask1] + 1.0) * 2\n    q[mask1, 0] = 0.25 * s1\n    q[mask1, 1] = (rotation_matrix[mask1, 2, 1] - rotation_matrix[mask1, 1, 2]) / s1\n    q[mask1, 2] = (rotation_matrix[mask1, 0, 2] - rotation_matrix[mask1, 2, 0]) / s1\n    q[mask1, 3] = (rotation_matrix[mask1, 1, 0] - rotation_matrix[mask1, 0, 1]) / s1\n    mask2 = ~mask1 & (rotation_matrix[:, 0, 0] > rotation_matrix[:, 1, 1]) & (rotation_matrix[:, 0, 0] > rotation_matrix[:, 2, 2])\n    s2 = torch.sqrt(1.0 + rotation_matrix[mask2, 0, 0] - rotation_matrix[mask2, 1, 1] - rotation_matrix[mask2, 2, 2]) * 2\n    q[mask2, 0] = (rotation_matrix[mask2, 2, 1] - rotation_matrix[mask2, 1, 2]) / s2\n    q[mask2, 1] = 0.25 * s2\n    q[mask2, 2] = (rotation_matrix[mask2, 0, 1] + rotation_matrix[mask2, 1, 0]) / s2\n    q[mask2, 3] = (rotation_matrix[mask2, 0, 2] + rotation_matrix[mask2, 2, 0]) / s2\n    mask3 = ~mask1 & ~mask2 & (rotation_matrix[:, 1, 1] > rotation_matrix[:, 2, 2])\n    s3 = torch.sqrt(1.0 + rotation_matrix[mask3, 1, 1] - rotation_matrix[mask3, 0, 0] - rotation_matrix[mask3, 2, 2]) * 2\n    q[mask3, 0] = (rotation_matrix[mask3, 0, 2] - rotation_matrix[mask3, 2, 0]) / s3\n    q[mask3, 1] = (rotation_matrix[mask3, 0, 1] + rotation_matrix[mask3, 1, 0]) / s3\n    q[mask3, 2] = 0.25 * s3\n    q[mask3, 3] = (rotation_matrix[mask3, 1, 2] + rotation_matrix[mask3, 2, 1]) / s3\n    mask4 = ~mask1 & ~mask2 & ~mask3\n    s4 = torch.sqrt(1.0 + rotation_matrix[mask4, 2, 2] - rotation_matrix[mask4, 0, 0] - rotation_matrix[mask4, 1, 1]) * 2\n    q[mask4, 0] = (rotation_matrix[mask4, 1, 0] - rotation_matrix[mask4, 0, 1]) / s4\n    q[mask4, 1] = (rotation_matrix[mask4, 0, 2] + rotation_matrix[mask4, 2, 0]) / s4\n    q[mask4, 2] = (rotation_matrix[mask4, 1, 2] + rotation_matrix[mask4, 2, 1]) / s4\n    q[mask4, 3] = 0.25 * s4\n    q = q / (torch.norm(q, dim=-1, keepdim=True) + 1e-10)\n    return q.reshape(batch_shape + (4,))\n\ndef matrix_to_quaternion_np(rotation_matrix):\n    mat = torch.from_numpy(rotation_matrix).contiguous().float()\n    return matrix_to_quaternion(mat).numpy()\n\ndef cont6d_to_quaternion(cont6d):\n    mat = cont6d_to_matrix(cont6d)\n    return matrix_to_quaternion(mat)\n\ndef cont6d_to_quaternion_np(cont6d):\n    q = torch.from_numpy(cont6d).contiguous().float()\n    return cont6d_to_quaternion(q).numpy()\n\ndef qpow(q0, t, dtype=torch.float):\n    q0 = qnormalize(q0)\n    theta0 = torch.acos(q0[..., 0])\n    mask = (theta0 <= 1e-09) * (theta0 >= -1e-09)\n    theta0 = (1 - mask) * theta0 + mask * 1e-09\n    v0 = q0[..., 1:] / torch.sin(theta0).view(-1, 1)\n    if isinstance(t, torch.Tensor):\n        q = torch.zeros(t.shape + q0.shape)\n        theta = t.view(-1, 1) * theta0.view(1, -1)\n    else:\n        q = torch.zeros(q0.shape)\n        theta = t * theta0\n    q[..., 0] = torch.cos(theta)\n    q[..., 1:] = v0 * torch.sin(theta).unsqueeze(-1)\n    return q.to(dtype)\n\ndef qslerp(q0, q1, t):\n    q0 = qnormalize(q0)\n    q1 = qnormalize(q1)\n    q_ = qpow(qmul(q1, qinv(q0)), t)\n    return qmul(q_, q0.contiguous().view(torch.Size([1] * len(t.shape)) + q0.shape).expand(t.shape + q0.shape).contiguous())\n\ndef qbetween(v0, v1):\n    assert v0.shape[-1] == 3, 'v0 must be of the shape (*, 3)'\n    assert v1.shape[-1] == 3, 'v1 must be of the shape (*, 3)'\n    v = torch.cross(v0, v1)\n    w = torch.sqrt((v0 ** 2).sum(dim=-1, keepdim=True) * (v1 ** 2).sum(dim=-1, keepdim=True)) + (v0 * v1).sum(dim=-1, keepdim=True)\n    return qnormalize(torch.cat([w, v], dim=-1))\n\ndef qbetween_np(v0, v1):\n    assert v0.shape[-1] == 3, 'v0 must be of the shape (*, 3)'\n    assert v1.shape[-1] == 3, 'v1 must be of the shape (*, 3)'\n    v0 = torch.from_numpy(v0).float()\n    v1 = torch.from_numpy(v1).float()\n    return qbetween(v0, v1).numpy()\n\ndef lerp(p0, p1, t):\n    if not isinstance(t, torch.Tensor):\n        t = torch.Tensor([t])\n    new_shape = t.shape + p0.shape\n    new_view_t = t.shape + torch.Size([1] * len(p0.shape))\n    new_view_p = torch.Size([1] * len(t.shape)) + p0.shape\n    p0 = p0.view(new_view_p).expand(new_shape)\n    p1 = p1.view(new_view_p).expand(new_shape)\n    t = t.view(new_view_t).expand(new_shape)\n    return p0 + t * (p1 - p0)",
    "utils/train_utils.py": "import copy\nimport os\nimport time\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.utils.data import DataLoader\nfrom typing import Optional, List, Union, Tuple\nfrom tqdm import tqdm\nfrom utils.wandb_logger import WandbLogger\n\ndef extract_prev_frame_features(frame: torch.Tensor) -> torch.Tensor:\n    B = frame.shape[0]\n    root_height = frame[:, 0:1]\n    root_vel = frame[:, 1:3]\n    root_rot6d = frame[:, 69:75]\n    prev_root = torch.cat([root_height, root_vel, root_rot6d], dim=-1)\n    joint_ric = frame[:, 6:69]\n    joint_rot6d = frame[:, 75:201]\n    joint_vel = frame[:, 204:267]\n    prev_joints = torch.cat([joint_ric, joint_rot6d, joint_vel], dim=-1)\n    return torch.cat([prev_root, prev_joints], dim=-1)\n\ndef extract_clean_target(frame: torch.Tensor) -> torch.Tensor:\n    root_height = frame[:, 0:1]\n    root_vel = frame[:, 1:3]\n    root_rot6d = frame[:, 69:75]\n    root_features = torch.cat([root_height, root_vel, root_rot6d], dim=-1)\n    joint_ric = frame[:, 6:69]\n    return torch.cat([root_features, joint_ric], dim=-1)\n\nclass EMAModel:\n\n    def __init__(self, model: nn.Module, decay: float=0.999):\n        self.decay = decay\n        self.model = copy.deepcopy(model)\n        for p in self.model.parameters():\n            p.requires_grad_(False)\n        self.model.eval()\n\n    def update(self, model: nn.Module) -> None:\n        with torch.no_grad():\n            for ema_p, p in zip(self.model.parameters(), model.parameters()):\n                ema_p.data.mul_(self.decay).add_(p.data, alpha=1 - self.decay)\n\n    def to(self, device: str) -> 'EMAModel':\n        self.model.to(device)\n        return self\n\ndef train(encoder: nn.Module, predictor: nn.Module, dataloader: DataLoader, num_epochs: int, save_dir: str, horizon: int=16, device: str='cuda', lr: float=0.0001, weight_decay: float=0.01, max_grad_norm: float=1.0, ema_decay: float=0.999, cfg_dropout: float=0.1, clip_encoder: Optional[nn.Module]=None, wandb_project: Optional[str]=None, wandb_run_name: Optional[str]=None, resume_from: Optional[str]=None) -> Tuple[EMAModel, EMAModel]:\n    os.makedirs(save_dir, exist_ok=True)\n    encoder.to(device)\n    predictor.to(device)\n    wandb_logger = None\n    if wandb_project:\n        config = {'lr': lr, 'weight_decay': weight_decay, 'max_grad_norm': max_grad_norm, 'ema_decay': ema_decay, 'num_epochs': num_epochs, 'horizon': horizon, 'cfg_dropout': cfg_dropout, 'batch_size': dataloader.batch_size, 'encoder_params': sum((p.numel() for p in encoder.parameters())), 'predictor_params': sum((p.numel() for p in predictor.parameters()))}\n        wandb_logger = WandbLogger(project=wandb_project, name=wandb_run_name, config=config)\n    encoder_ema = EMAModel(encoder, decay=ema_decay).to(device)\n    predictor_ema = EMAModel(predictor, decay=ema_decay).to(device)\n    params = list(encoder.parameters()) + list(predictor.parameters())\n    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)\n    use_amp = device.startswith('cuda')\n    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)\n    global_step = 0\n    best_loss = float('inf')\n    best_epoch = -1\n    start_epoch = 0\n    if resume_from is not None and os.path.exists(resume_from):\n        print(f'Resuming from checkpoint: {resume_from}')\n        checkpoint = torch.load(resume_from, map_location=device)\n        encoder.load_state_dict(checkpoint['encoder'])\n        predictor.load_state_dict(checkpoint['predictor'])\n        encoder_ema.model.load_state_dict(checkpoint['encoder_ema'])\n        predictor_ema.model.load_state_dict(checkpoint['predictor_ema'])\n        optimizer.load_state_dict(checkpoint['optimizer'])\n        scaler.load_state_dict(checkpoint['scaler'])\n        start_epoch = checkpoint.get('epoch', 0) + 1\n        global_step = checkpoint.get('global_step', 0)\n        best_loss = checkpoint.get('best_loss', float('inf'))\n        best_epoch = checkpoint.get('best_epoch', -1)\n        print(f'Resumed from epoch {start_epoch}, step {global_step}')\n    print(f'Training for {num_epochs} epochs with horizon={horizon}')\n    print(f'Encoder params: {sum((p.numel() for p in encoder.parameters())):,}')\n    print(f'Predictor params: {sum((p.numel() for p in predictor.parameters())):,}')\n    encoder.train()\n    predictor.train()\n\n    def save_checkpoint(filename: str, loss: float, epoch: int):\n        path = os.path.join(save_dir, filename)\n        torch.save({'encoder': encoder.state_dict(), 'predictor': predictor.state_dict(), 'encoder_ema': encoder_ema.model.state_dict(), 'predictor_ema': predictor_ema.model.state_dict(), 'optimizer': optimizer.state_dict(), 'scaler': scaler.state_dict(), 'epoch': epoch, 'global_step': global_step, 'loss': loss, 'horizon': horizon, 'best_loss': best_loss, 'best_epoch': best_epoch}, path)\n        print(f'Saved checkpoint: {path}')\n    try:\n        for epoch in tqdm(range(start_epoch, num_epochs), desc='Training', unit='epoch'):\n            epoch_loss = 0.0\n            num_batches = 0\n            pbar = tqdm(dataloader, desc=f'Epoch {epoch}', leave=False, unit='batch')\n            batch_start_time = time.time()\n            for batch in pbar:\n                motion = batch['motion'].to(device)\n                B, T, _ = motion.shape\n                if 'text_clip' in batch:\n                    text = batch['text_clip'].to(device)\n                elif 'captions' in batch and clip_encoder is not None:\n                    captions = batch['captions']\n                    with torch.no_grad():\n                        text = clip_encoder(captions)\n                elif 'captions' in batch:\n                    raise ValueError(\"Raw captions provided but no clip_encoder. Pass clip_encoder to train() or provide pre-encoded 'text_clip' in dataset.\")\n                else:\n                    raise ValueError(\"No text input found. Batch must contain 'text_clip' or 'captions'.\")\n                lengths = batch.get('lengths', torch.full((B,), T, device=device, dtype=torch.long))\n                min_length = int(lengths.min().item())\n                horizon = min(horizon, min_length - 1)\n                max_start = max(1, min_length - horizon - 1)\n                start_idx = torch.randint(0, max_start, (1,)).item()\n                end_idx = start_idx + horizon\n                hist = motion[:, start_idx:end_idx]\n                target_frames = motion[:, start_idx + 1:end_idx + 1]\n                text_input = text if torch.rand(1).item() > cfg_dropout else None\n                optimizer.zero_grad(set_to_none=True)\n                use_amp = device.startswith('cuda')\n                amp_dtype = torch.float32\n                if use_amp:\n                    amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16\n                with torch.amp.autocast(device, dtype=amp_dtype, enabled=use_amp):\n                    contexts = encoder(text=text_input, input_features=hist, batch_size=B)\n                    num_pred_frames = min(contexts.shape[1], target_frames.shape[1])\n                    pred_contexts = contexts[:, -num_pred_frames:]\n                    prev_frames = hist[:, -num_pred_frames:]\n                    target_frames = target_frames[:, -num_pred_frames:]\n                    B, N, J, D = pred_contexts.shape\n                    contexts_flat = pred_contexts.reshape(B * N, J, D)\n                    prev_flat = prev_frames.reshape(B * N, 271)\n                    targets_flat = target_frames.reshape(B * N, 271)\n                    prev_features = extract_prev_frame_features(prev_flat)\n                    clean_targets = extract_clean_target(targets_flat)\n                    t = torch.rand(B * N, device=device)\n                    noise = torch.randn_like(clean_targets)\n                    x_t = t.view(B * N, 1) * clean_targets + (1 - t.view(B * N, 1)) * noise\n                    pred = predictor(history_features=contexts_flat, noise_level=t, noisy_target=x_t, prev_frame_features=prev_features)\n                    target_v = clean_targets - noise\n                    loss = F.mse_loss(pred, target_v)\n                scaler.scale(loss).backward()\n                scaler.unscale_(optimizer)\n                grad_norm = torch.nn.utils.clip_grad_norm_(params, max_grad_norm)\n                scaler.step(optimizer)\n                scaler.update()\n                encoder_ema.update(encoder)\n                predictor_ema.update(predictor)\n                batch_time = time.time() - batch_start_time\n                batch_start_time = time.time()\n                gpu_memory_allocated = 0.0\n                gpu_memory_reserved = 0.0\n                if torch.cuda.is_available():\n                    gpu_memory_allocated = torch.cuda.memory_allocated() / 1000000000.0\n                    gpu_memory_reserved = torch.cuda.memory_reserved() / 1000000000.0\n                pbar.set_postfix({'loss': f'{loss.item():.4f}', 'lr': f'{lr:.2e}'})\n                if wandb_logger is not None:\n                    wandb_logger.log({'train/loss': loss.item(), 'train/lr': lr, 'train/epoch': epoch, 'train/grad_norm': grad_norm.item() if hasattr(grad_norm, 'item') else grad_norm, 'train/batch_time': batch_time, 'train/samples_per_sec': B / batch_time if batch_time > 0 else 0, 'train/horizon': horizon, 'train/num_pred_frames': num_pred_frames, 'system/gpu_memory_allocated_gb': gpu_memory_allocated, 'system/gpu_memory_reserved_gb': gpu_memory_reserved}, step=global_step)\n                if global_step % 100 == 0:\n                    tqdm.write(f'[Epoch {epoch}] [Step {global_step}] loss={loss.item():.6f} lr={lr:.2e}')\n                epoch_loss += loss.item()\n                num_batches += 1\n                global_step += 1\n            pbar.close()\n            avg_epoch_loss = epoch_loss / max(1, num_batches)\n            tqdm.write(f'==> End of Epoch {epoch}: Avg Loss = {avg_epoch_loss:.6f}')\n            if wandb_logger is not None:\n                wandb_logger.log({'epoch/avg_loss': avg_epoch_loss, 'epoch/num': epoch}, step=global_step)\n            save_checkpoint('latest.pt', avg_epoch_loss, epoch)\n            if avg_epoch_loss < best_loss:\n                tqdm.write(f'New best model! (Loss: {best_loss:.6f} -> {avg_epoch_loss:.6f})')\n                best_loss = avg_epoch_loss\n                best_epoch = epoch\n                save_checkpoint('best.pt', avg_epoch_loss, epoch)\n    except KeyboardInterrupt:\n        tqdm.write('Training interrupted. Saving emergency checkpoint...')\n        save_checkpoint('latest_interrupted.pt', 0.0, epoch)\n        tqdm.write('Done.')\n    if wandb_logger is not None:\n        wandb_logger.log_summary({'best_loss': best_loss, 'best_epoch': best_epoch})\n        wandb_logger.finish()\n    return (encoder_ema, predictor_ema)\n\ndef validate(encoder: nn.Module, predictor: nn.Module, dataloader: DataLoader, horizon: int, device: str='cuda', num_batches: int=10, clip_encoder: Optional[nn.Module]=None) -> dict:\n    encoder.eval()\n    predictor.eval()\n    total_loss = 0.0\n    total_samples = 0\n    with torch.no_grad():\n        for i, batch in enumerate(dataloader):\n            if i >= num_batches:\n                break\n            motion = batch['motion'].to(device)\n            B, T, _ = motion.shape\n            if 'text_clip' in batch:\n                text = batch['text_clip'].to(device)\n            elif 'captions' in batch and clip_encoder is not None:\n                captions = batch['captions']\n                text = clip_encoder(captions)\n            elif 'captions' in batch:\n                raise ValueError(\"Raw captions provided but no clip_encoder. Pass clip_encoder to validate() or provide pre-encoded 'text_clip'.\")\n            else:\n                raise ValueError(\"No text input found. Batch must contain 'text_clip' or 'captions'.\")\n            lengths = batch.get('lengths', torch.full((B,), T, device=device, dtype=torch.long))\n            min_length = int(lengths.min().item())\n            if min_length <= horizon + 1:\n                continue\n            max_start = max(1, min_length - horizon - 1)\n            start_idx = torch.randint(0, max_start, (1,)).item()\n            end_idx = min(start_idx + horizon, min_length - 1)\n            hist = motion[:, start_idx:end_idx]\n            target_frame = motion[:, end_idx]\n            prev_features = extract_prev_frame_features(hist[:, -1])\n            clean_target = extract_clean_target(target_frame)\n            context = encoder(text=text, input_features=hist, batch_size=B)\n            t = torch.rand(B, device=device)\n            noise = torch.randn_like(clean_target)\n            x_t = t.view(B, 1) * clean_target + (1 - t.view(B, 1)) * noise\n            pred = predictor(history_features=context, noise_level=t, noisy_target=x_t, prev_frame_features=prev_features)\n            target_v = clean_target - noise\n            loss = F.mse_loss(pred, target_v)\n            total_loss += loss.item() * B\n            total_samples += B\n    encoder.train()\n    predictor.train()\n    return {'val_loss': total_loss / max(1, total_samples)}\n\ndef generate_free_running(encoder: nn.Module, predictor: nn.Module, text: torch.Tensor, num_frames: int, num_flow_steps: int=10, device: str='cuda', initial_features: Optional[torch.Tensor]=None, dataset_type: str='t2m') -> torch.Tensor:\n    from .motion_utils import IncrementalFeatureExtractor, features_to_positions\n    encoder.eval()\n    predictor.eval()\n    B = text.shape[0]\n    if initial_features is None:\n        history = encoder.null_history.expand(B, 1, -1).clone()\n    else:\n        history = initial_features.unsqueeze(1)\n    extractor = IncrementalFeatureExtractor(dataset_type=dataset_type, device=torch.device(device))\n    init_positions = features_to_positions(history[:, -1], dataset_type=dataset_type)\n    extractor.initialize(init_positions)\n    current_frame_271d = history[:, -1]\n    current_positions = init_positions\n    generated_frames = []\n    with torch.no_grad():\n        for frame_idx in range(num_frames):\n            context = encoder(text=text, input_features=history, batch_size=B)\n            prev_features = extract_prev_frame_features(history[:, -1])\n            x_t = torch.randn(B, 72, device=device)\n            dt = 1.0 / num_flow_steps\n            for step in range(num_flow_steps):\n                t = torch.full((B,), step * dt, device=device)\n                v = predictor(history_features=context, noise_level=t, noisy_target=x_t, prev_frame_features=prev_features)\n                x_t = x_t + v * dt\n            generated_frames.append(x_t)\n            new_frame_271d, new_positions = extractor.process_flow_output(x_t, current_frame_271d)\n            current_frame_271d = new_frame_271d\n            current_positions = new_positions\n            history = new_frame_271d.unsqueeze(1)\n    encoder.train()\n    predictor.train()\n    return torch.stack(generated_frames, dim=1)",
    "utils/text_encoder.py": '"""\nText encoding utility using CLIP model from Hugging Face Transformers.\n"""\n\nimport torch\nfrom transformers import CLIPTokenizer, CLIPTextModel\nfrom typing import List, Union\n\n\nclass CLIPEncoder(torch.nn.Module):\n    """\n    Utility class to encode text captions using Microsoft\'s CLIP model.\n    By default, uses \'openai/clip-vit-base-patch32\' which produces 512D embeddings.\n\n    For texts longer than 77 tokens, uses chunk-and-average approach to preserve\n    all text content.\n    """\n\n    def __init__(\n        self,\n        model_name: str = "openai/clip-vit-base-patch32",\n        max_length: int = 77,\n    ):\n        super().__init__()\n\n        self.max_length = max_length\n\n        print(f"Loading CLIP model \'{model_name}\'...")\n        self.tokenizer = CLIPTokenizer.from_pretrained(model_name)\n        self.model = CLIPTextModel.from_pretrained(model_name)\n        self.model.eval()\n\n        # Freeze CLIP parameters\n        for param in self.model.parameters():\n            param.requires_grad = False\n\n    @torch.no_grad()\n    def forward(self, text: Union[str, List[str]]) -> torch.Tensor:\n        """\n        Encode a list of captions or a single caption into embeddings.\n\n        For texts longer than max_length tokens, splits into chunks and averages\n        the embeddings to preserve all text content.\n\n        Args:\n            text: A single string or a list of strings.\n\n        Returns:\n            embeddings: (B, 1, 512) tensor containing the pooled embeddings.\n        """\n        if isinstance(text, str):\n            text = [text]\n\n        # Determine device dynamically\n        device = next(self.model.parameters()).device\n\n        embeddings_list = []\n\n        for caption in text:\n            # Tokenize without padding/truncation to check length\n            tokens = self.tokenizer(caption, return_tensors="pt", truncation=False)\n            input_ids = tokens["input_ids"][0]\n            seq_len = len(input_ids)\n\n            if seq_len <= self.max_length:\n                # Short text: encode directly\n                inputs = self.tokenizer(\n                    caption,\n                    padding=True,\n                    truncation=True,\n                    max_length=self.max_length,\n                    return_tensors="pt",\n                ).to(device)\n                output = self.model(**inputs)\n                embedding = output.pooler_output  # (1, 512)\n            else:\n                # Long text: chunk and average\n                chunk_embeddings = []\n\n                # Split into overlapping chunks\n                # Start from position 1 to skip BOS token for chunks\n                stride = self.max_length - 2  # Leave room for BOS and EOS\n\n                for start_idx in range(0, seq_len - 1, stride):\n                    end_idx = min(start_idx + self.max_length - 1, seq_len - 1)\n\n                    # Extract chunk tokens (keep BOS at start, EOS at end)\n                    if start_idx == 0:\n                        chunk_ids = input_ids[: end_idx + 1]\n                    else:\n                        # Add BOS token at the beginning\n                        bos_token = torch.tensor([self.tokenizer.bos_token_id or 49406])\n                        chunk_ids = torch.cat(\n                            [bos_token, input_ids[start_idx : end_idx + 1]]\n                        )\n\n                    # Ensure EOS token at the end\n                    if chunk_ids[-1] != self.tokenizer.eos_token_id:\n                        eos_token = torch.tensor([self.tokenizer.eos_token_id or 49407])\n                        chunk_ids = torch.cat([chunk_ids, eos_token])\n\n                    # Truncate if still too long\n                    if len(chunk_ids) > self.max_length:\n                        chunk_ids = chunk_ids[: self.max_length - 1]\n                        eos_token = torch.tensor([self.tokenizer.eos_token_id or 49407])\n                        chunk_ids = torch.cat([chunk_ids, eos_token])\n\n                    # Encode chunk\n                    attention_mask = torch.ones_like(chunk_ids)\n                    inputs = {\n                        "input_ids": chunk_ids.unsqueeze(0).to(device),\n                        "attention_mask": attention_mask.unsqueeze(0).to(device),\n                    }\n                    output = self.model(**inputs)\n                    chunk_embeddings.append(output.pooler_output)\n\n                # Average all chunk embeddings\n                embedding = torch.stack(chunk_embeddings, dim=0).mean(dim=0)  # (1, 512)\n\n            embeddings_list.append(embedding)\n\n        # Stack all embeddings\n        embeddings = torch.cat(embeddings_list, dim=0)  # (B, 512)\n\n        return embeddings.unsqueeze(1)  # (B, 1, 512)\n\n    @property\n    def embedding_dim(self) -> int:\n        """Output dimension of the CLIP text model."""\n        return self.model.config.hidden_size\n',
    "utils/wandb_logger.py": "import os\nimport sys\nfrom datetime import datetime\nfrom typing import Optional, Dict, Any\ntry:\n    import wandb\n    WANDB_AVAILABLE = True\nexcept ImportError:\n    WANDB_AVAILABLE = False\n    wandb = None\n\ndef is_kaggle_environment() -> bool:\n    return os.path.exists('/kaggle') or 'kaggle' in sys.executable.lower()\n\ndef get_kaggle_secret(secret_name: str) -> Optional[str]:\n    if not is_kaggle_environment():\n        return None\n    try:\n        from kaggle_secrets import UserSecretsClient\n        user_secrets = UserSecretsClient()\n        return user_secrets.get_secret(secret_name)\n    except Exception:\n        return None\n\nclass WandbLogger:\n\n    def __init__(self, project: str, name: Optional[str]=None, config: Optional[Dict[str, Any]]=None, kaggle_secret_name: str='WANDB_API_KEY', enabled: bool=True):\n        self.project = project\n        self.config = config or {}\n        self.enabled = enabled and WANDB_AVAILABLE\n        self.run = None\n        if name is None:\n            self.name = 'motion-generation-buet'\n        else:\n            self.name = name\n        if not self.enabled:\n            if not WANDB_AVAILABLE:\n                print('[WandbLogger] wandb not installed. Logging disabled.')\n            elif not enabled:\n                print('[WandbLogger] Logging disabled by user.')\n            return\n        self._authenticate(kaggle_secret_name)\n        try:\n            self.run = wandb.init(project=project, entity='motion-generation-buet', config=config, reinit=True)\n            print(f'[WandbLogger] Initialized run: {self.run.name}')\n            print(f'[WandbLogger] View at: {self.run.url}')\n        except Exception as e:\n            print(f'[WandbLogger] Failed to initialize: {e}')\n            self.enabled = False\n\n    def _authenticate(self, secret_name: str) -> None:\n        api_key = os.environ.get('WANDB_API_KEY')\n        if api_key is None:\n            api_key = get_kaggle_secret(secret_name)\n        if api_key:\n            try:\n                wandb.login(key=api_key)\n                print('[WandbLogger] Authenticated successfully.')\n            except Exception as e:\n                print(f'[WandbLogger] Authentication failed: {e}')\n        else:\n            print('[WandbLogger] No API key found. Using existing login or anonymous mode.')\n\n    def log(self, metrics: Dict[str, Any], step: Optional[int]=None) -> None:\n        if not self.enabled or self.run is None:\n            return\n        try:\n            wandb.log(metrics, step=step)\n        except Exception as e:\n            print(f'[WandbLogger] Failed to log metrics: {e}')\n\n    def log_model(self, path: str, name: str, description: Optional[str]=None) -> None:\n        if not self.enabled or self.run is None:\n            return\n        try:\n            artifact = wandb.Artifact(name, type='model', description=description)\n            artifact.add_file(path)\n            self.run.log_artifact(artifact)\n            print(f'[WandbLogger] Logged model artifact: {name}')\n        except Exception as e:\n            print(f'[WandbLogger] Failed to log model: {e}')\n\n    def log_summary(self, metrics: Dict[str, Any]) -> None:\n        if not self.enabled or self.run is None:\n            return\n        try:\n            for key, value in metrics.items():\n                wandb.run.summary[key] = value\n        except Exception as e:\n            print(f'[WandbLogger] Failed to log summary: {e}')\n\n    def finish(self) -> None:\n        if not self.enabled or self.run is None:\n            return\n        try:\n            wandb.finish()\n            print('[WandbLogger] Run finished.')\n        except Exception as e:\n            print(f'[WandbLogger] Failed to finish run: {e}')\n\n    def __enter__(self) -> 'WandbLogger':\n        return self\n\n    def __exit__(self, exc_type, exc_val, exc_tb) -> None:\n        self.finish()",
}

for filepath, content in FILES.items():
    path = Path(filepath)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"Created {filepath}")

# 3D Human Motion Generation Pipeline

This notebook implements a complete pipeline for generating 3D human motion animations using:

- **Motion History Encoder**: Encodes motion context with text conditioning
- **Flow Matching Predictor**: Generates motion using flow matching
- **CLIP Text Encoder**: Encodes text prompts for conditioning

**Input**: Text description + optional motion history
**Output**: 3D joint positions (22 joints × 3D coordinates)


## Setup and Imports


In [20]:
# Install dependencies (uncomment if running on Kaggle/Colab)
# !pip install torch transformers matplotlib tqdm

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm

from config import Config
from models import MotionHistoryEncoder, FlowMatchingPredictor, HumanMotionGenerator
from utils.dataset import Text2MotionDataset, text2motion_collate_fn, create_dataloader
from utils.text_encoder import CLIPEncoder
from utils.train_utils import train
from utils.utils import visualize_motion

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load configuration
config = Config()
config.device = device
print(f"Dataset path: {config.dataset_path}")
print(f"Checkpoint dir: {config.checkpoint_dir}")

In [21]:
# Create dataset and dataloader
print("Loading dataset...")
config.dataset_path = Path(
    "/kaggle/input/notebooks/mustafamuhaimin/3d-human-motion-generation/dataset/humanml3d-subset/"
)
config.checkpoint_dir = Path(
    "/kaggle/input/notebooks/mustafamuhaimin/3d-human-motion-generation/checkpoints/"
)

config.batch_size = 1
dataloader = create_dataloader(config, split="train", shuffle=True)

print(f"Number of batches: {len(dataloader)}")

# Show a sample batch
sample_batch = next(iter(dataloader))
print(f"\nSample batch:")
print(f"  Captions: {len(sample_batch['captions'])} samples")
print(f"  Motion shape: {sample_batch['motion'].shape}")  # (B, T, 263)
print(f"  Joints shape: {sample_batch['joints'].shape}")  # (B, T, 22, 3)
print(f"  Text embeddings shape: {sample_batch['text_clip'].shape}")  # (B, 512)
print(f"  Lengths shape: {sample_batch['lengths'].shape}")  # (B,)
print(f"\nSample caption: '{sample_batch['captions'][0]}'")

## Load Trained Model for Generation


In [22]:
!ls -lah checkpoints

In [23]:
# Load the best checkpoint for generation
checkpoint_path = config.checkpoint_dir / "best.pt"

generator = HumanMotionGenerator.load_from_checkpoint(
    checkpoint_path=checkpoint_path, config=config, device=str(device)
)

print(f"✅ Loaded model from {checkpoint_path}")
print("Model ready for generation!")

## Initialize CLIP Text Encoder


In [24]:
# Initialize CLIP encoder for text-to-embedding conversion
clip_encoder = CLIPEncoder(model_name="openai/clip-vit-base-patch32")
clip_encoder.to(device)

print("✅ CLIP encoder initialized")

## Generate Motion from Text Prompts

Generate 3D human motion animations from text descriptions.


In [25]:
dataset_iter = iter(dataloader)
d = next(dataset_iter)
d["captions"][0:1]

In [26]:
d.keys()

In [27]:
d["motion"][0:1].shape

In [32]:
# Enter your custom text prompt
print(f"Generating motion for: '{d['captions']}'")
input_features = d["motion"][:, 0:20, :]
# Encode text to CLIP embedding
motions = []
with torch.no_grad():
    text_embedding = d["text_clip"].to(device)  # (1, 512)

joint_positions = generator.generate_sequence(
    text=text_embedding,
    # input_features=input_features,
    num_frames=100,
    num_steps=25,
    guidance_scale=1,
    dataset_type="t2m",
)

In [33]:
joint_positions[0, :].cpu().shape, d["joints"][0, :].cpu().shape

In [34]:
joint_positions[0, 0]

In [31]:
# Visualize
motions_np = joint_positions[0, :].cpu().numpy()
ani = visualize_motion(motions_np, title=d["captions"], fps=20)
ani

In [ ]:
# Define text prompts for generation
text_prompts = [
    "a person walks forward",
    "a person is running",
    "a person jumps up",
]

# Generation parameters
num_steps = 25  # Number of flow matching steps (higher = better quality, slower)
guidance_scale = 2.5  # CFG scale (1.0 = no guidance, 2.5-3.5 recommended)

print(f"Generating {len(text_prompts)} motions...")
print(f"Flow matching steps: {num_steps}")
print(f"Guidance scale: {guidance_scale}\n")

generated_motions = []

for i, prompt in enumerate(text_prompts):
    print(f"[{i+1}/{len(text_prompts)}] Generating: '{prompt}'")

    # Encode text to CLIP embedding
    with torch.no_grad():
        text_embedding = clip_encoder([prompt]).to(device)  # (1, 512)

    joint_positions = generator.generate_sequence(
        text=text_embedding,
        num_frames=100,
        num_steps=10,
        guidance_scale=2.5,
        dataset_type="t2m",
    )

    # Store generated motion
    generated_motions.append(joint_positions.cpu().numpy())
    print(f"  ✓ Generated shape: {joint_positions.shape}\n")

print("✅ All motions generated successfully!")

## Visualize Generated Motions

Visualize each generated motion as an animated 3D skeleton.


In [ ]:
# Visualize first motion: "a person walks forward"
motion_idx = 0
joints = generated_motions[motion_idx][0]  # Remove batch dimension
prompt = text_prompts[motion_idx]

print(f"Visualizing: '{prompt}'")
print(f"Motion shape: {joints.shape}")

ani = visualize_motion(joints, title=prompt, fps=20)
ani

In [ ]:
# Visualize second motion: "a person is running"
motion_idx = 1
joints = generated_motions[motion_idx][0]
prompt = text_prompts[motion_idx]

print(f"Visualizing: '{prompt}'")
ani = visualize_motion(joints, title=prompt, fps=20)
ani

In [ ]:
# # Visualize third motion: "a person jumps up"
# motion_idx = 2
# joints = generated_motions[motion_idx]
# prompt = text_prompts[motion_idx]

# print(f"Visualizing: '{prompt}'")
# ani = visualize_motion(joints, title=prompt, fps=20)
# ani

In [ ]:
# # Visualize fourth motion: "a person waves their hand"
# motion_idx = 3
# joints = generated_motions[motion_idx]
# prompt = text_prompts[motion_idx]

# print(f"Visualizing: '{prompt}'")
# ani = visualize_motion(joints, title=prompt, fps=20)
# ani

In [ ]:
# # Visualize fifth motion: "a person sits down"
# motion_idx = 4
# joints = generated_motions[motion_idx]
# prompt = text_prompts[motion_idx]

# print(f"Visualizing: '{prompt}'")
# ani = visualize_motion(joints, title=prompt, fps=20)
# ani

## Save Generated Motions


In [ ]:
# # Create output directory
# output_dir = config.output_path / "generated_motions"
# output_dir.mkdir(parents=True, exist_ok=True)

# # Save each generated motion
# for i, (motion, prompt) in enumerate(zip(generated_motions, text_prompts)):
#     # Save as numpy file
#     filename = f"motion_{i:03d}_{prompt.replace(' ', '_')[:30]}.npy"
#     filepath = output_dir / filename
#     np.save(filepath, motion)
#     print(f"Saved: {filepath}")

# print(f"\n✅ All motions saved to {output_dir}")

## Custom Text Prompt Generation

Try your own text prompts!


In [ ]:
# # Enter your custom text prompt
# custom_prompt = "a person does a backflip"  # Change this to your desired motion

# print(f"Generating motion for: '{custom_prompt}'")

# # Encode text to CLIP embedding
# motions = []
# with torch.no_grad():
#     text_embedding = clip_encoder([prompt]).to(device)  # (1, 512)

# joint_positions = generator.generate_sequence(
#     text=text_embedding,
#     num_frames=200,
#     num_steps=10,
#     guidance_scale=2.5,
#     dataset_type="t2m",
# )

# # Visualize
# motions_np = joint_positions.cpu().numpy()
# ani = visualize_motion(motions_np, title=custom_prompt, fps=20)
# ani